In [28]:
import pandas as pd
from linearmodels.panel import PanelOLS
from linearmodels.panel import RandomEffects
from linearmodels.panel import compare
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import statsmodels.formula.api as smf
from statsmodels.stats.diagnostic import acorr_ljungbox
from scipy.stats import chi2
from pathlib import Path
import statsmodels.api as sm
import numpy as np
from scipy.stats import chi2

In [29]:
# 데이터 불러오기
df = pd.read_csv("/Volumes/ORICO/Climate risk/CBRE_NOAA 2007-2024.csv")

df.head()

/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1919/384946503.py:2: DtypeWarning: Columns (4,6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Volumes/ORICO/Climate risk/CBRE_NOAA 2007-2024.csv")


,Year of Data,PropertyID,City,State,Zip,YearOpened,Rooms,Occupancy,ADR,RevPar,...,Fire_events,Tropical_events,Other_events,Meteorological_damage,Hydrological_damage,Climatological_damage,Geophysical_damage,Fire_damage,Tropical_damage,Other_damage
0,2007,101,Atlanta,GA,30326,1974.0,230,75.2%,$122.98,$92.45,...,7.0,2.0,0.0,134151100.0,327000.0,505000000.0,0.0,21701000.0,0.0,0.0
1,2007,111,Atlanta,GA,30344,1973.0,378,67.0%,$98.28,$65.86,...,7.0,2.0,0.0,134151100.0,327000.0,505000000.0,0.0,21701000.0,0.0,0.0
2,2007,118,Arlington,VA,22202,1972.0,280,73.2%,$131.86,$96.55,...,2.0,0.0,0.0,3436000.0,154000.0,17355000.0,0.0,85000.0,0.0,0.0
3,2007,123,Gaithersburg,MD,20879,1971.0,300,63.3%,$93.31,$59.10,...,0.0,0.0,0.0,349000.0,11000.0,5000.0,0.0,0.0,0.0,0.0
4,2007,124,Denver,CO,80207,1973.0,561,78.5%,$81.04,$63.65,...,7.0,0.0,0.0,5425750.0,3502000.0,20000.0,50000.0,771000.0,0.0,0.0


In [30]:
print(df.dtypes)

Year of Data               int64
PropertyID                 int64
City                      object
State                     object
Zip                       object
                          ...   
Climatological_damage    float64
Geophysical_damage       float64
Fire_damage              float64
Tropical_damage          float64
Other_damage             float64
Length: 199, dtype: object


In [31]:
# ChainSegment 간단히 그룹화
def simplify_chain(segment):
    if segment in ['Luxury', 'Upper Upscale']:
        return 'High-End'
    elif segment in ['Upscale', 'Upper Midscale']:
        return 'Mid-Range'
    elif segment in ['Midscale', 'Economy']:
        return 'Budget'

df['ChainGroup'] = df['ChainSegment'].apply(simplify_chain)

# ChainGroup: Mid-Range를 reference
df['ChainGroup'] = pd.Categorical(df['ChainGroup'],
                                  categories=['Mid-Range', 'High-End', 'Budget'],
                                  ordered=False)


# 새 그룹으로 더미 생성
dummies = pd.get_dummies(df['ChainGroup'], drop_first=True)
df = pd.concat([df, dummies], axis=1)

In [32]:
print(df['ChainSegment'].value_counts())
print(df['ChainGroup'].value_counts())


ChainSegment
Upscale           34062
Upper Midscale    23092
Upper Upscale     17481
Economy           11576
Luxury             5180
Midscale           4744
Name: count, dtype: int64
ChainGroup
Mid-Range    57154
High-End     22661
Budget       16320
Name: count, dtype: int64


In [33]:
# noaa_region9 정리 및 카테고리화
df["noaa_region9"] = df["noaa_region9"].astype(str).str.strip()
df.loc[df["noaa_region9"].isin(["", "NaN", "nan", "None"]), "noaa_region9"] = np.nan

conus_regions = [
    "Northeast","Southeast","South","Ohio Valley","Upper Midwest",
    "Northern Rockies & Plains","Southwest","Northwest","West"
]

# 기준범주(reference) 자동 선택: CONUS 중 최빈 → 없으면 전체 최빈 → 없으면 'West'
mode_conus = df.loc[df["noaa_region9"].isin(conus_regions), "noaa_region9"].mode(dropna=True)
if not mode_conus.empty:
    baseline = mode_conus.iloc[0]
else:
    mode_all = df["noaa_region9"].mode(dropna=True)
    baseline = mode_all.iloc[0] if not mode_all.empty else "West"

cats = [baseline] + sorted([c for c in df["noaa_region9"].dropna().unique() if c != baseline])
df["noaa_region9"] = pd.Categorical(df["noaa_region9"], categories=cats, ordered=False)

print("Reference(기준) region =", baseline)

# 3) (선택) 원-핫 더미 만들기 (drop_first=True) → 회귀 X행렬로 직접 쓰고 싶을 때
noaa_dummies = pd.get_dummies(df["noaa_region9"], prefix="noaa", drop_first=True)


df = pd.concat([df, noaa_dummies], axis=1)


print(noaa_dummies.columns)
print(df.shape)
print(df[['noaa_region9'] + list(noaa_dummies.columns)].head())



Reference(기준) region = Southeast
Index(['noaa_Midwest', 'noaa_Northeast', 'noaa_Northern Plains',
       'noaa_Northwest', 'noaa_Pacific', 'noaa_South', 'noaa_Southwest',
       'noaa_West'],
      dtype='object')
(96358, 210)
  noaa_region9  noaa_Midwest  noaa_Northeast  noaa_Northern Plains  \
0    Southeast         False           False                 False   
1    Southeast         False           False                 False   
2    Southeast         False           False                 False   
3    Northeast         False            True                 False   
4    Southwest         False           False                 False   

   noaa_Northwest  noaa_Pacific  noaa_South  noaa_Southwest  noaa_West  
0           False         False       False           False      False  
1           False         False       False           False      False  
2           False         False       False           False      False  
3           False         False       False           False 

In [34]:
def clean_numeric(s):
    s = s.astype(str)
    s = s.str.replace(",", "", regex=False)
    s = s.str.replace("$", "", regex=False)
    s = s.str.strip()
    s = s.replace(["", "nan", "-", "None"], np.nan)
    return pd.to_numeric(s, errors="coerce")


# 🔹 CBRE 주요 재무변수 클린
cols_to_clean = [
    'ITTotalExpense','ITSandW','ITSystemExpenseOperDept','ITTotalOtherExpense',
    'ITTotalLabor','ITCostofServicePhone','ITCostofServiceInternet',
    'ITSystemExpenseUndistribDept','TotalLabor','MaintTotalExp','UtilTotalExp',
    'Insurance','RevPar','ADR','GrossOperatingProfit','EBITDA',
    'TotalHotelRevenue','TotalSAndW','TotalBenefits','SandMTotalExpense',
    'MktingTotalExp','MktingAdvertising','TotalMgmtFee','BaseMgmtFee',
    'IncentMgmtFee','FandBTotalRevenue','PropertyTax',
    
    # ✅ 추가됨
    'Available',   # for Goppar
    'YearOpened'   # age 계산 안정화용 (있으면)
]

for c in cols_to_clean:
    if c in df.columns:
        df[c] = clean_numeric(df[c])



# 🔹 Derived metrics
df['Goppar'] = df['GrossOperatingProfit'] / df['Available']
df['F&BRatio'] = df['FandBTotalRevenue'] / df['TotalHotelRevenue']
df['ITService'] = df['ITCostofServicePhone'] + df['ITCostofServiceInternet']
df['ITSystem'] = df['ITSystemExpenseOperDept'] + df['ITSystemExpenseUndistribDept']

# 🔹 age
df['age'] = df['Year of Data'] - pd.to_numeric(df['YearOpened'], errors="coerce")


# 🔹 NOAA 피해/건수 클린
noaa_cols = [
    'Meteorological_events','Meteorological_damage','Hydrological_events',
    'Hydrological_damage','Climatological_events','Climatological_damage',
    'Geophysical_events','Geophysical_damage','Fire_events','Fire_damage',
    'Tropical_events','Tropical_damage','Other_events','Other_damage'
]

for c in noaa_cols:
    if c in df.columns:
        df[c] = clean_numeric(df[c])


In [35]:
#로그 및 Asinh

#log 
df['ITTotalExpense_log'] = np.log1p(df["ITTotalExpense"])
df['ITTotalLabor_log'] = np.log1p(df["ITTotalLabor"])
df['ITService_log'] = np.log1p(df["ITService"])
df['ITSystem_log'] = np.log1p(df["ITSystem"])
df['TotalLabor_log'] = np.log1p(df["TotalLabor"])
df['ITTotalLabor_log'] = np.log1p(df["ITTotalLabor"])

df['MaintTotalExp_log'] = np.log1p(df["MaintTotalExp"])
df['UtilTotalExp_log'] = np.log1p(df["UtilTotalExp"])
df['Insurance_log'] = np.log1p(df["Insurance"])
df['RevPar_log'] = np.log1p(df["RevPar"])
df['PropertyTax_log'] = np.log1p(df["PropertyTax"])
df['SandMTotalExpense_log'] = np.log1p(df["SandMTotalExpense"])
df['MktingTotalExp_log'] = np.log1p(df["MktingTotalExp"])
df['TotalMgmtFee_log'] = np.log1p(df["TotalMgmtFee"])


In [36]:
#기후리스크 건수와 피해액 합수 변수 생성


df['Meteorological_events_log'] = np.log1p(df["Meteorological_events"])
df['Meteorological_damage_log'] = np.log1p(df["Meteorological_damage"])
df['Hydrological_events_log'] = np.log1p(df["Hydrological_events"])
df['Hydrological_damage_log'] = np.log1p(df["Hydrological_damage"])
df['Climatological_events_log'] = np.log1p(df["Climatological_events"])
df['Climatological_damage_log'] = np.log1p(df["Climatological_damage"])
df['Geophysical_events_log'] = np.log1p(df["Geophysical_events"])
df['Geophysical_damage_log'] = np.log1p(df["Geophysical_damage"])
df['Fire_events_log'] = np.log1p(df["Fire_events"])
df['Fire_damage_log'] = np.log1p(df["Fire_damage"])
df['Tropical_events_log'] = np.log1p(df["Tropical_events"])
df['Tropical_damage_log'] = np.log1p(df["Tropical_damage"])
df['Other_events_log'] = np.log1p(df["Other_events"])
df['Other_damage_log'] = np.log1p(df["Other_damage"])
#df['ZipClimate_Meteorological_events_log'] = np.log1p(df["ZipClimate_Meteorological_events"])
#df['ZipClimate_Meteorological_damage_log'] = np.log1p(df["ZipClimate_Meteorological_damage"])
#df['ZipClimate_Hydrological_events_log'] = np.log1p(df["ZipClimate_Hydrological_events"])
#df['ZipClimate_Hydrological_damage_log'] = np.log1p(df["ZipClimate_Hydrological_damage"])



# 결합형

df["Meteorological_risk_log"] = (
    df["Meteorological_events_log"] + df["Meteorological_damage_log"]
) / 2

df["Hydrological_risk_log"] = (
    df["Hydrological_events_log"] + df["Hydrological_damage_log"]
) / 2

df["Climatological_risk_log"] = (
    df["Climatological_events_log"] + df["Climatological_damage_log"]
) / 2

df["Geophysical_risk_log"] = (
    df["Geophysical_events_log"] + df["Geophysical_damage_log"]
) / 2

df["Fire_risk_log"] = (
    df["Fire_events_log"] + df["Fire_damage_log"]
) / 2

df["Tropical_risk_log"] = (
    df["Tropical_events_log"] + df["Tropical_damage_log"]
) / 2

df["Other_risk_log"] = (
    df["Other_events_log"] + df["Other_damage_log"]
) / 2

#df["Zip_Meteorological_risk_log"] = (
#    df["ZipClimate_Meteorological_events_log"] + df["ZipClimate_Meteorological_damage_log"]
#) / 2

#df["Zip_Hydrological_risk_log"] = (
#    df["ZipClimate_Hydrological_events_log"] + df["ZipClimate_Hydrological_damage_log"]
#) / 2

In [37]:
df.head()

,Year of Data,PropertyID,City,State,Zip,YearOpened,Rooms,Occupancy,ADR,RevPar,...,Tropical_damage_log,Other_events_log,Other_damage_log,Meteorological_risk_log,Hydrological_risk_log,Climatological_risk_log,Geophysical_risk_log,Fire_risk_log,Tropical_risk_log,Other_risk_log
0,2007,101,Atlanta,GA,30326,1974.0,230,75.2%,122.98,92.45,...,0.0,0.0,0.0,12.265794,7.547807,11.609061,0.000000,9.486155,0.549306,0.0
1,2007,111,Atlanta,GA,30344,1973.0,378,67.0%,98.28,65.86,...,0.0,0.0,0.0,12.265794,7.547807,11.609061,0.000000,9.486155,0.549306,0.0
2,2007,118,Arlington,VA,22202,1972.0,280,73.2%,131.86,96.55,...,0.0,0.0,0.0,10.291604,7.540104,10.290707,0.000000,6.224515,0.000000,0.0
3,2007,123,Gaithersburg,MD,20879,1971.0,300,63.3%,93.31,59.10,...,0.0,0.0,0.0,8.830335,5.851818,6.173017,0.000000,0.000000,0.000000,0.0
4,2007,124,Denver,CO,80207,1973.0,561,78.5%,81.04,63.65,...,0.0,0.0,0.0,10.723419,9.391209,6.599687,6.103046,7.817443,0.000000,0.0


In [11]:
df = df.set_index(['PropertyID', 'Year of Data'])

In [38]:
df = df.sort_values(by=['PropertyID', 'Year of Data'])

# Lag 원 변수 생성
df['lag_ITTotalExpense'] = df.groupby('PropertyID')['ITTotalExpense'].shift(1)
df['lag_TotalLabor'] = df.groupby('PropertyID')['TotalLabor'].shift(1)
df['lag_ITTotalLabor'] = df.groupby('PropertyID')['ITTotalLabor'].shift(1)
df['lag_ITService'] = df.groupby('PropertyID')['ITService'].shift(1)
df['lag_ITSystem'] = df.groupby('PropertyID')['ITSystem'].shift(1)

df['lag_MaintTotalExp'] = df.groupby('PropertyID')['MaintTotalExp'].shift(1)
df['lag_UtilTotalExp'] = df.groupby('PropertyID')['UtilTotalExp'].shift(1)
df['lag_Insurance'] = df.groupby('PropertyID')['Insurance'].shift(1)
df['lag_SandMTotalExpense'] = df.groupby('PropertyID')['SandMTotalExpense'].shift(1)
df['lag_MktingTotalExp'] = df.groupby('PropertyID')['MktingTotalExp'].shift(1)

df['lag_TotalMgmtFee'] = df.groupby('PropertyID')['TotalMgmtFee'].shift(1)
df['lag_PropertyTax'] = df.groupby('PropertyID')['PropertyTax'].shift(1)
df['lag_Rooms'] = df.groupby('PropertyID')['Rooms'].shift(1)
df['lag_age'] = df.groupby('PropertyID')['age'].shift(1)
df['lag_MktingTotalExp'] = df.groupby('PropertyID')['MktingTotalExp'].shift(1)

df["lag_Meteorological_risk_log"] = df.groupby('PropertyID')['Meteorological_risk_log'].shift(1)
df["lag_Hydrological_risk_log"] = df.groupby('PropertyID')['Hydrological_risk_log'].shift(1)
df["lag_Climatological_risk_log"] = df.groupby('PropertyID')['Climatological_risk_log'].shift(1)
df["lag_Geophysical_risk_log"] = df.groupby('PropertyID')['Geophysical_risk_log'].shift(1)
df["lag_Fire_risk_log"] = df.groupby('PropertyID')['Fire_risk_log'].shift(1)
df["lag_Tropical_risk_log"] = df.groupby('PropertyID')['Tropical_risk_log'].shift(1)
df["lag_Other_risk_log"] = df.groupby('PropertyID')['Other_risk_log'].shift(1)
#df["lag_Zip_Meteorological_risk_log"] = df.groupby('PropertyID')['Zip_Meteorological_risk_log'].shift(1)
#df["lag_Zip_Hydrological_risk_log"] = df.groupby('PropertyID')['Zip_Hydrological_risk_log'].shift(1)

#df['lag_ITTotalOtherExpense'] = df.groupby('PropertyID')['ITTotalOtherExpense'].shift(1)'

/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1919/3213558728.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['lag_ITService'] = df.groupby('PropertyID')['ITService'].shift(1)
/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1919/3213558728.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['lag_ITSystem'] = df.groupby('PropertyID')['ITSystem'].shift(1)
/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1919/3213558728.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is u

In [39]:
#lag된 기후 리스크 교호항 생성

df['lag_ITxMeteo_risk'] = df['lag_ITTotalExpense'] * df["lag_Meteorological_risk_log"]
df['lag_ITxHydro_risk'] = df['lag_ITTotalExpense'] * df["lag_Hydrological_risk_log"]
df['lag_ITxClimato_risk'] = df['lag_ITTotalExpense'] * df["lag_Climatological_risk_log"]
df['lag_ITxGeophy_risk'] = df['lag_ITTotalExpense'] * df["lag_Geophysical_risk_log"]
df['lag_ITxFire_risk'] = df['lag_ITTotalExpense'] * df["lag_Fire_risk_log"]
df['lag_ITxTropi_risk'] = df['lag_ITTotalExpense'] * df["lag_Tropical_risk_log"]
df['lag_ITxOther_risk'] = df['lag_ITTotalExpense'] * df["lag_Other_risk_log"]
df['lag_ITLaborxMeteo_risk'] = df['lag_ITTotalLabor'] * df["lag_Meteorological_risk_log"]
df['lag_ITLaborxHydro_risk'] = df['lag_ITTotalLabor'] * df["lag_Hydrological_risk_log"]
df['lag_ITLaborxClimato_risk'] = df['lag_ITTotalLabor'] * df["lag_Climatological_risk_log"]
df['lag_ITLaborxGeophy_risk'] = df['lag_ITTotalLabor'] * df["lag_Geophysical_risk_log"]
df['lag_ITLaborxFire_risk'] = df['lag_ITTotalLabor'] * df["lag_Fire_risk_log"]
df['lag_ITLaborxTropi_risk'] = df['lag_ITTotalLabor'] * df["lag_Tropical_risk_log"]
df['lag_ITLaborxOther_risk'] = df['lag_ITTotalLabor'] * df["lag_Other_risk_log"]
df['lag_ITServxMeteo_risk'] = df['lag_ITService'] * df["lag_Meteorological_risk_log"]
df['lag_ITServxHydro_risk'] = df['lag_ITService'] * df["lag_Hydrological_risk_log"]
df['lag_ITServxClimato_risk'] = df['lag_ITService'] * df["lag_Climatological_risk_log"]
df['lag_ITServxGeophy_risk'] = df['lag_ITService'] * df["lag_Geophysical_risk_log"]
df['lag_ITServxFire_risk'] = df['lag_ITService'] * df["lag_Fire_risk_log"]
df['lag_ITServxTropi_risk'] = df['lag_ITService'] * df["lag_Tropical_risk_log"]
df['lag_ITServxOther_risk'] = df['lag_ITService'] * df["lag_Other_risk_log"]
df['lag_ITSysxMeteo_risk'] = df['lag_ITSystem'] * df["lag_Meteorological_risk_log"]
df['lag_ITSysxHydro_risk'] = df['lag_ITSystem'] * df["lag_Hydrological_risk_log"]
df['lag_ITSysxClimato_risk'] = df['lag_ITSystem'] * df["lag_Climatological_risk_log"]
df['lag_ITSysxGeophy_risk'] = df['lag_ITSystem'] * df["lag_Geophysical_risk_log"]
df['lag_ITSysxFire_risk'] = df['lag_ITSystem'] * df["lag_Fire_risk_log"]
df['lag_ITSysxTropi_risk'] = df['lag_ITSystem'] * df["lag_Tropical_risk_log"]
df['lag_ITSysxOther_risk'] = df['lag_ITSystem'] * df["lag_Other_risk_log"]
df['lag_LaborxMeteo_risk'] = df['lag_TotalLabor'] * df["lag_Meteorological_risk_log"]
df['lag_LaborxHydro_risk'] = df['lag_TotalLabor'] * df["lag_Hydrological_risk_log"]
df['lag_LaborxClimato_risk'] = df['lag_TotalLabor'] * df["lag_Climatological_risk_log"]
df['lag_LaborxGeophy_risk'] = df['lag_TotalLabor'] * df["lag_Geophysical_risk_log"]
df['lag_LaborxFire_risk'] = df['lag_TotalLabor'] * df["lag_Fire_risk_log"]
df['lag_LaborxTropi_risk'] = df['lag_TotalLabor'] * df["lag_Tropical_risk_log"]
df['lag_LaborxOther_risk'] = df['lag_TotalLabor'] * df["lag_Other_risk_log"]

/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1919/3982829646.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['lag_ITxMeteo_risk'] = df['lag_ITTotalExpense'] * df["lag_Meteorological_risk_log"]
/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1919/3982829646.py:4: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['lag_ITxHydro_risk'] = df['lag_ITTotalExpense'] * df["lag_Hydrological_risk_log"]
/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1919/3982829646.py:5: PerformanceWarning: DataFra

In [40]:
df.head()

,Year of Data,PropertyID,City,State,Zip,YearOpened,Rooms,Occupancy,ADR,RevPar,...,lag_ITSysxFire_risk,lag_ITSysxTropi_risk,lag_ITSysxOther_risk,lag_LaborxMeteo_risk,lag_LaborxHydro_risk,lag_LaborxClimato_risk,lag_LaborxGeophy_risk,lag_LaborxFire_risk,lag_LaborxTropi_risk,lag_LaborxOther_risk
0,2007,101,Atlanta,GA,30326,1974.0,230,75.2%,122.98,92.45,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8729,2009,101,Atlanta,GA,30326,1974.0,230,64.7%,106.11,68.68,...,NaN,NaN,NaN,3.893285e+07,2.395749e+07,3.684831e+07,0.0,3.011000e+07,1.743552e+06,0.000000e+00
12498,2010,101,Atlanta,GA,30326,1974.0,230,72.0%,102.70,73.91,...,NaN,NaN,NaN,3.389268e+07,3.148290e+07,0.000000e+00,0.0,2.151131e+07,9.930896e+05,1.188453e+07
16957,2011,101,Atlanta,GA,30326.0,1974.0,230,68.3%,105.49,72.06,...,NaN,NaN,NaN,3.384558e+07,2.518296e+07,3.688204e+06,0.0,2.089158e+07,0.000000e+00,2.659406e+06
22005,2012,101,Atlanta,GA,30326,1974.0,230,73.7%,111.47,82.18,...,NaN,NaN,NaN,3.570600e+07,2.089779e+07,3.122713e+07,0.0,2.600394e+07,1.013100e+06,2.352345e+06


In [18]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler


# ===========================================================
# ✅ MixedLM with PCA-based Tropical Risk
#  - Risk = PCA1(log1p(events), log1p(damage))
#  - Risk_z = z-score of PCA score
#  - IT: within/between decomposition (z-scored)
#  - Interactions: IT_within×Risk_z, IT_mean_hotel×Risk_z
#  - FE: ChainSegment, Region9, Year
#  - RE: (1) Random intercept  (2) Random intercept + random slope(IT_within)
# ===========================================================

# ------------------------------
# 0) 안전한 reset_index (ID가 index였을 수 있음)
# ------------------------------
df = df.reset_index()

# ------------------------------
# 1) 변수 설정 / rename
# ------------------------------
ID     = "PropertyID"
REGION = "noaa_region9"
SEG    = "ChainSegment"

if "Year of Data" in df.columns and "Year" not in df.columns:
    df = df.rename(columns={"Year of Data": "Year"})
YEAR = "Year"

Y  = "RevPar_log"
IT = "ITTotalExpense"



# 1️⃣ log return 생성
df["RevPar_return"] = df.groupby(ID)["RevPar_log"].diff()

# 2️⃣ 5년 rolling std
df["RevPar_vol_5y"] = (
    df.groupby(ID)["RevPar_return"]
      .rolling(window=5, min_periods=3)
      .std()
      .reset_index(level=0, drop=True)
)

# PCA 구성에 사용할 원천 변수(원래 이미 만들어둔 log 변수 사용)
EV_LOG = "Tropical_events_log"
DMG_LOG = "Tropical_damage_log"

CONTROLS = [
    "TotalLabor","MaintTotalExp","UtilTotalExp",
    "Insurance","SandMTotalExpense","MktingTotalExp",
    "TotalMgmtFee","PropertyTax","Rooms","age"
]

# ------------------------------
# 2) 변환 함수 (money vars)
# ------------------------------
def safe_transform(series, use_asinh_if_negative=True):
    s = pd.to_numeric(series, errors="coerce")
    if use_asinh_if_negative and (s.dropna() < 0).any():
        return np.arcsinh(s)
    else:
        return np.log1p(s.clip(lower=0))

money_vars = [
    IT,
    "TotalLabor","MaintTotalExp","UtilTotalExp","Insurance",
    "SandMTotalExpense","MktingTotalExp","TotalMgmtFee",
    "PropertyTax","Rooms"
]

for col in money_vars:
    if col in df.columns and f"tr_{col}" not in df.columns:
        df[f"tr_{col}"] = safe_transform(df[col], use_asinh_if_negative=True)

IT_term = f"tr_{IT}"

# controls term 구성 (age는 그대로)
CONTROLS_term = []
for c in CONTROLS:
    if c == "age":
        CONTROLS_term.append("age")
    elif c in money_vars:
        CONTROLS_term.append(f"tr_{c}")
    else:
        CONTROLS_term.append(c)

# ------------------------------
# 3) 분석용 컬럼 구성 (PCA용 변수 포함)
# ------------------------------
base_cols = [Y, ID, REGION, SEG, YEAR, IT_term, EV_LOG, DMG_LOG] + CONTROLS_term
d = df[base_cols].dropna().copy()

# Year는 int 유지
d[YEAR] = pd.to_numeric(d[YEAR], errors="coerce")
d = d.dropna(subset=[YEAR])
d[YEAR] = d[YEAR].astype(int)

# 카테고리
d[SEG] = d[SEG].astype("category")
d[REGION] = d[REGION].astype("category")

# ------------------------------
# 4) PCA Risk 생성 (샘플 d에서 학습 → d에 부여)
#    - 표준화 후 PCA 추천 (둘의 스케일/분산 차이를 제거)
# ------------------------------
X = d[[EV_LOG, DMG_LOG]].values
X_std = StandardScaler(with_mean=True, with_std=True).fit_transform(X)

pca = PCA(n_components=1, random_state=0)
risk_pca = pca.fit_transform(X_std).ravel()

# 방향성(부호) 정리: 해석 편의를 위해 "events+damage 증가" 방향이 +가 되도록
# (PC는 부호가 임의라서 고정해주는 게 논문에 유리)
if np.corrcoef(risk_pca, d[EV_LOG].values)[0,1] < 0:
    risk_pca = -risk_pca
    pca.components_ = -pca.components_

d["Risk_pca"] = risk_pca

# Risk_z (z-score)
d["Risk_z"] = (d["Risk_pca"] - d["Risk_pca"].mean()) / d["Risk_pca"].std(ddof=0)

print("PCA weights (events, damage):", pca.components_)
print("Explained variance ratio:", pca.explained_variance_ratio_)

# ------------------------------
# 5) IT within / between 분해 + z-score
# ------------------------------
d["IT_mean_hotel"] = d.groupby(ID)[IT_term].transform("mean")
d["IT_within"] = d[IT_term] - d["IT_mean_hotel"]

for col in ["IT_mean_hotel", "IT_within"]:
    d[col] = (d[col] - d[col].mean()) / d[col].std(ddof=0)

# ------------------------------
# 6) Interaction (within/between 각각)
# ------------------------------
d["ITw_x_R"] = d["IT_within"] * d["Risk_z"]
d["ITb_x_R"] = d["IT_mean_hotel"] * d["Risk_z"]

# ------------------------------
# 7) controls 표준화 (age 제외) - 선택(권장)
# ------------------------------
z_cols = [c for c in CONTROLS_term if c != "age"]
for col in z_cols:
    if col in d.columns:
        sd = d[col].std(ddof=0)
        if sd > 0:
            d[col] = (d[col] - d[col].mean()) / sd

# ------------------------------
# 8) MixedLM 식 정의 (FE 포함)
# ------------------------------
rhs_terms = [
    "IT_within",
    "IT_mean_hotel",
    "Risk_z",
    "ITw_x_R",
    "ITb_x_R"
] + CONTROLS_term + [f"C({SEG})", f"C({REGION})", f"C({YEAR})"]

rhs = " + ".join(rhs_terms)

# ------------------------------
# 9) Step 1: Random Intercept
# ------------------------------
m1 = smf.mixedlm(
    f"{Y} ~ {rhs}",
    data=d,
    groups=d[ID],
    re_formula="1"
)
res1 = m1.fit(method="lbfgs", reml=True)
print(res1.summary())

# ------------------------------
# 10) Step 2: Random Intercept + Random Slope (IT_within)
# ------------------------------
m2 = smf.mixedlm(
    f"{Y} ~ {rhs}",
    data=d,
    groups=d[ID],
    re_formula="1 + IT_within"
)

try:
    res2 = m2.fit(method="lbfgs", reml=True)
except Exception as e1:
    print("LBFGS failed, trying Powell. Error:", e1)
    res2 = m2.fit(method="powell", reml=True)

print(res2.summary())

print("\nModel Comparison:")
print(f"  Log-Likelihood (RI)    : {res1.llf:.2f}")
print(f"  Log-Likelihood (RI+RS) : {res2.llf:.2f}")

PCA weights (events, damage): [[0.70710678 0.70710678]]
Explained variance ratio: [0.96242549]
                     Mixed Linear Model Regression Results
Model:                     MixedLM        Dependent Variable:        RevPar_log
No. Observations:          70867          Method:                    REML      
No. Groups:                10228          Scale:                     0.0162    
Min. group size:           1              Log-Likelihood:            30044.1006
Max. group size:           18             Converged:                 Yes       
Mean group size:           6.9                                                 
-------------------------------------------------------------------------------
                                   Coef.  Std.Err.    z     P>|z| [0.025 0.975]
-------------------------------------------------------------------------------
Intercept                           4.505    0.012  383.107 0.000  4.482  4.528
C(ChainSegment)[T.Luxury]           0.069    0

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                     Mixed Linear Model Regression Results
Model:                     MixedLM        Dependent Variable:        RevPar_log
No. Observations:          70867          Method:                    REML      
No. Groups:                10228          Scale:                     0.0141    
Min. group size:           1              Log-Likelihood:            31321.6405
Max. group size:           18             Converged:                 Yes       
Mean group size:           6.9                                                 
-------------------------------------------------------------------------------
                                   Coef.  Std.Err.    z     P>|z| [0.025 0.975]
-------------------------------------------------------------------------------
Intercept                           4.512    0.012  388.452 0.000  4.490  4.535
C(ChainSegment)[T.Luxury]           0.063    0.012    5.218 0.000  0.040  0.087
C(ChainSegment)[T.Midscale]         0.015    0.011    1.373 0

In [37]:
#PCA INDEXING FOR TROPICAL RISK

# ===========================================================
# ✅ MixedLM with PCA-based Tropical Risk
#  - Risk = PCA1(log1p(events), log1p(damage))
#  - Risk_z = z-score of PCA score
#  - IT: within/between decomposition (z-scored)
#  - Interactions: IT_within×Risk_z, IT_mean_hotel×Risk_z
#  - FE: ChainSegment, Region9, Year
#  - RE: (1) Random intercept  (2) Random intercept + random slope(IT_within)
# ===========================================================

# ------------------------------
# 0) 안전한 reset_index (ID가 index였을 수 있음)
# ------------------------------
df = df.reset_index()

# ------------------------------
# 1) 변수 설정 / rename
# ------------------------------
ID     = "PropertyID"
REGION = "noaa_region9"
SEG    = "ChainSegment"

if "Year of Data" in df.columns and "Year" not in df.columns:
    df = df.rename(columns={"Year of Data": "Year"})
YEAR = "Year"

Y  = "RevPar_log"
IT = "ITTotalExpense"

# PCA 구성에 사용할 원천 변수(원래 이미 만들어둔 log 변수 사용)
EV_LOG = "Tropical_events_log"
DMG_LOG = "Tropical_damage_log"

CONTROLS = [
    "TotalLabor","MaintTotalExp","UtilTotalExp",
    "Insurance","SandMTotalExpense","MktingTotalExp",
    "TotalMgmtFee","PropertyTax","Rooms","age"
]

# ------------------------------
# 2) 변환 함수 (money vars)
# ------------------------------
def safe_transform(series, use_asinh_if_negative=True):
    s = pd.to_numeric(series, errors="coerce")
    if use_asinh_if_negative and (s.dropna() < 0).any():
        return np.arcsinh(s)
    else:
        return np.log1p(s.clip(lower=0))

money_vars = [
    IT,
    "TotalLabor","MaintTotalExp","UtilTotalExp","Insurance",
    "SandMTotalExpense","MktingTotalExp","TotalMgmtFee",
    "PropertyTax","Rooms"
]

for col in money_vars:
    if col in df.columns and f"tr_{col}" not in df.columns:
        df[f"tr_{col}"] = safe_transform(df[col], use_asinh_if_negative=True)

IT_term = f"tr_{IT}"

# controls term 구성 (age는 그대로)
CONTROLS_term = []
for c in CONTROLS:
    if c == "age":
        CONTROLS_term.append("age")
    elif c in money_vars:
        CONTROLS_term.append(f"tr_{c}")
    else:
        CONTROLS_term.append(c)

# ------------------------------
# 3) 분석용 컬럼 구성 (PCA용 변수 포함)
# ------------------------------
base_cols = [Y, ID, REGION, SEG, YEAR, IT_term, EV_LOG, DMG_LOG] + CONTROLS_term
d = df[base_cols].dropna().copy()

# Year는 int 유지
d[YEAR] = pd.to_numeric(d[YEAR], errors="coerce")
d = d.dropna(subset=[YEAR])
d[YEAR] = d[YEAR].astype(int)

# 카테고리
d[SEG] = d[SEG].astype("category")
d[REGION] = d[REGION].astype("category")

# ------------------------------
# 4) PCA Risk 생성 (샘플 d에서 학습 → d에 부여)
#    - 표준화 후 PCA 추천 (둘의 스케일/분산 차이를 제거)
# ------------------------------
X = d[[EV_LOG, DMG_LOG]].values
X_std = StandardScaler(with_mean=True, with_std=True).fit_transform(X)

pca = PCA(n_components=1, random_state=0)
risk_pca = pca.fit_transform(X_std).ravel()

# 방향성(부호) 정리: 해석 편의를 위해 "events+damage 증가" 방향이 +가 되도록
# (PC는 부호가 임의라서 고정해주는 게 논문에 유리)
if np.corrcoef(risk_pca, d[EV_LOG].values)[0,1] < 0:
    risk_pca = -risk_pca
    pca.components_ = -pca.components_

d["Risk_pca"] = risk_pca

# Risk_z (z-score)
d["Risk_z"] = (d["Risk_pca"] - d["Risk_pca"].mean()) / d["Risk_pca"].std(ddof=0)

print("PCA weights (events, damage):", pca.components_)
print("Explained variance ratio:", pca.explained_variance_ratio_)

# ------------------------------
# 5) IT within / between 분해 + z-score
# ------------------------------
d["IT_mean_hotel"] = d.groupby(ID)[IT_term].transform("mean")
d["IT_within"] = d[IT_term] - d["IT_mean_hotel"]

for col in ["IT_mean_hotel", "IT_within"]:
    d[col] = (d[col] - d[col].mean()) / d[col].std(ddof=0)

# ------------------------------
# 6) Interaction (within/between 각각)
# ------------------------------
d["ITw_x_R"] = d["IT_within"] * d["Risk_z"]
d["ITb_x_R"] = d["IT_mean_hotel"] * d["Risk_z"]

# ------------------------------
# 7) controls 표준화 (age 제외) - 선택(권장)
# ------------------------------
z_cols = [c for c in CONTROLS_term if c != "age"]
for col in z_cols:
    if col in d.columns:
        sd = d[col].std(ddof=0)
        if sd > 0:
            d[col] = (d[col] - d[col].mean()) / sd

# ------------------------------
# 8) MixedLM 식 정의 (FE 포함)
# ------------------------------
rhs_terms = [
    "IT_within",
    "IT_mean_hotel",
    "Risk_z",
    "ITw_x_R",
    "ITb_x_R"
] + CONTROLS_term + [f"C({SEG})", f"C({REGION})", f"C({YEAR})"]

rhs = " + ".join(rhs_terms)

# ------------------------------
# 9) Step 1: Random Intercept
# ------------------------------
m1 = smf.mixedlm(
    f"{Y} ~ {rhs}",
    data=d,
    groups=d[ID],
    re_formula="1"
)
res1 = m1.fit(method="lbfgs", reml=True)
print(res1.summary())

# ------------------------------
# 10) Step 2: Random Intercept + Random Slope (IT_within)
# ------------------------------
m2 = smf.mixedlm(
    f"{Y} ~ {rhs}",
    data=d,
    groups=d[ID],
    re_formula="1 + IT_within"
)

try:
    res2 = m2.fit(method="lbfgs", reml=True)
except Exception as e1:
    print("LBFGS failed, trying Powell. Error:", e1)
    res2 = m2.fit(method="powell", reml=True)

print(res2.summary())

print("\nModel Comparison:")
print(f"  Log-Likelihood (RI)    : {res1.llf:.2f}")
print(f"  Log-Likelihood (RI+RS) : {res2.llf:.2f}")

# ===========================================================
# ✅ Nakagawa R² (prediction-based, random slope 안전 버전)
# ===========================================================

def mixedlm_r2_pred(res):
    """
    Prediction-based Nakagawa R²
    Works for random intercept + random slope models
    """
    m = res.model

    # Fixed effect fitted values (Xβ)
    yhat_fix = m.exog @ res.fe_params.values

    # Full fitted values (Xβ + Zu)
    yhat_full = np.asarray(res.fittedvalues)

    # Random effect contribution (Zu)
    Zu = yhat_full - yhat_fix

    # Variance components
    var_fix   = np.var(yhat_fix, ddof=1)
    var_re    = np.var(Zu, ddof=1)
    var_resid = res.scale

    denom = var_fix + var_re + var_resid

    R2_marginal   = var_fix / denom
    R2_conditional = (var_fix + var_re) / denom

    return {
        "R2_marginal": R2_marginal,
        "R2_conditional": R2_conditional,
        "var_fixed": var_fix,
        "var_random": var_re,
        "var_residual": var_resid
    }

print("\n--- R² (Nakagawa, prediction-based) ---")

r2_res1 = mixedlm_r2_pred(res1)
print("\nRandom Intercept Model:")
for k, v in r2_res1.items():
    print(f"{k}: {v:.4f}")

r2_res2 = mixedlm_r2_pred(res2)
print("\nRandom Intercept + Random Slope Model:")
for k, v in r2_res2.items():
    print(f"{k}: {v:.4f}")

PCA weights (events, damage): [[0.70710678 0.70710678]]
Explained variance ratio: [0.96242549]
                     Mixed Linear Model Regression Results
Model:                     MixedLM        Dependent Variable:        RevPar_log
No. Observations:          70867          Method:                    REML      
No. Groups:                10228          Scale:                     0.0162    
Min. group size:           1              Log-Likelihood:            30044.1006
Max. group size:           18             Converged:                 Yes       
Mean group size:           6.9                                                 
-------------------------------------------------------------------------------
                                   Coef.  Std.Err.    z     P>|z| [0.025 0.975]
-------------------------------------------------------------------------------
Intercept                           4.505    0.012  383.107 0.000  4.482  4.528
C(ChainSegment)[T.Luxury]           0.069    0

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                     Mixed Linear Model Regression Results
Model:                     MixedLM        Dependent Variable:        RevPar_log
No. Observations:          70867          Method:                    REML      
No. Groups:                10228          Scale:                     0.0141    
Min. group size:           1              Log-Likelihood:            31321.6405
Max. group size:           18             Converged:                 Yes       
Mean group size:           6.9                                                 
-------------------------------------------------------------------------------
                                   Coef.  Std.Err.    z     P>|z| [0.025 0.975]
-------------------------------------------------------------------------------
Intercept                           4.512    0.012  388.452 0.000  4.490  4.535
C(ChainSegment)[T.Luxury]           0.063    0.012    5.218 0.000  0.040  0.087
C(ChainSegment)[T.Midscale]         0.015    0.011    1.373 0

In [43]:
#PCA INDEXING FOR FIRE RISK

# ===========================================================
# ✅ MixedLM with PCA-based Tropical Risk
#  - Risk = PCA1(log1p(events), log1p(damage))
#  - Risk_z = z-score of PCA score
#  - IT: within/between decomposition (z-scored)
#  - Interactions: IT_within×Risk_z, IT_mean_hotel×Risk_z
#  - FE: ChainSegment, Region9, Year
#  - RE: (1) Random intercept  (2) Random intercept + random slope(IT_within)
# ===========================================================

# ------------------------------
# 0) 안전한 reset_index (ID가 index였을 수 있음)
# ------------------------------
# Safe reset (한 번만)
df = df.reset_index(drop=True)

# ------------------------------
# 1) 변수 설정 / rename
# ------------------------------
ID     = "PropertyID"
REGION = "noaa_region9"
SEG    = "ChainSegment"

if "Year of Data" in df.columns and "Year" not in df.columns:
    df = df.rename(columns={"Year of Data": "Year"})
YEAR = "Year"

Y  = "RevPar_log"
IT = "ITTotalExpense"

# PCA 구성에 사용할 원천 변수(원래 이미 만들어둔 log 변수 사용)
EV_LOG = "Fire_events_log"
DMG_LOG = "Fire_damage_log"

CONTROLS = [
    "TotalLabor","MaintTotalExp","UtilTotalExp",
    "Insurance","SandMTotalExpense","MktingTotalExp",
    "TotalMgmtFee","PropertyTax","Rooms","age"
]

# ------------------------------
# 2) 변환 함수 (money vars)
# ------------------------------
def safe_transform(series, use_asinh_if_negative=True):
    s = pd.to_numeric(series, errors="coerce")
    if use_asinh_if_negative and (s.dropna() < 0).any():
        return np.arcsinh(s)
    else:
        return np.log1p(s.clip(lower=0))

money_vars = [
    IT,
    "TotalLabor","MaintTotalExp","UtilTotalExp","Insurance",
    "SandMTotalExpense","MktingTotalExp","TotalMgmtFee",
    "PropertyTax","Rooms"
]

for col in money_vars:
    if col in df.columns and f"tr_{col}" not in df.columns:
        df[f"tr_{col}"] = safe_transform(df[col], use_asinh_if_negative=True)

IT_term = f"tr_{IT}"

# controls term 구성 (age는 그대로)
CONTROLS_term = []
for c in CONTROLS:
    if c == "age":
        CONTROLS_term.append("age")
    elif c in money_vars:
        CONTROLS_term.append(f"tr_{c}")
    else:
        CONTROLS_term.append(c)

# ------------------------------
# 3) 분석용 컬럼 구성 (PCA용 변수 포함)
# ------------------------------
base_cols = [Y, ID, REGION, SEG, YEAR, IT_term, EV_LOG, DMG_LOG] + CONTROLS_term
d = df[base_cols].dropna().copy()

# Year는 int 유지
d[YEAR] = pd.to_numeric(d[YEAR], errors="coerce")
d = d.dropna(subset=[YEAR])
d[YEAR] = d[YEAR].astype(int)

# 카테고리
d[SEG] = d[SEG].astype("category")
d[REGION] = d[REGION].astype("category")

# ------------------------------
# 4) PCA Risk 생성 (샘플 d에서 학습 → d에 부여)
#    - 표준화 후 PCA 추천 (둘의 스케일/분산 차이를 제거)
# ------------------------------
X = d[[EV_LOG, DMG_LOG]].values
X_std = StandardScaler(with_mean=True, with_std=True).fit_transform(X)

pca = PCA(n_components=1, random_state=0)
risk_pca = pca.fit_transform(X_std).ravel()

# 방향성(부호) 정리: 해석 편의를 위해 "events+damage 증가" 방향이 +가 되도록
# (PC는 부호가 임의라서 고정해주는 게 논문에 유리)
if np.corrcoef(risk_pca, d[EV_LOG].values)[0,1] < 0:
    risk_pca = -risk_pca
    pca.components_ = -pca.components_

d["Risk_pca"] = risk_pca

# Risk_z (z-score)
d["Risk_z"] = (d["Risk_pca"] - d["Risk_pca"].mean()) / d["Risk_pca"].std(ddof=0)

print("PCA weights (events, damage):", pca.components_)
print("Explained variance ratio:", pca.explained_variance_ratio_)

# ------------------------------
# 5) IT within / between 분해 + z-score
# ------------------------------
d["IT_mean_hotel"] = d.groupby(ID)[IT_term].transform("mean")
d["IT_within"] = d[IT_term] - d["IT_mean_hotel"]

for col in ["IT_mean_hotel", "IT_within"]:
    d[col] = (d[col] - d[col].mean()) / d[col].std(ddof=0)

# ------------------------------
# 6) Interaction (within/between 각각)
# ------------------------------
d["ITw_x_R"] = d["IT_within"] * d["Risk_z"]
d["ITb_x_R"] = d["IT_mean_hotel"] * d["Risk_z"]

# ------------------------------
# 7) controls 표준화 (age 제외) - 선택(권장)
# ------------------------------
z_cols = [c for c in CONTROLS_term if c != "age"]
for col in z_cols:
    if col in d.columns:
        sd = d[col].std(ddof=0)
        if sd > 0:
            d[col] = (d[col] - d[col].mean()) / sd

# ------------------------------
# 8) MixedLM 식 정의 (FE 포함)
# ------------------------------
rhs_terms = [
    "IT_within",
    "IT_mean_hotel",
    "Risk_z",
    "ITw_x_R",
    "ITb_x_R"
] + CONTROLS_term + [f"C({SEG})", f"C({REGION})", f"C({YEAR})"]

rhs = " + ".join(rhs_terms)

# ------------------------------
# 9) Step 1: Random Intercept
# ------------------------------
m1 = smf.mixedlm(
    f"{Y} ~ {rhs}",
    data=d,
    groups=d[ID],
    re_formula="1"
)
res1 = m1.fit(method="lbfgs", reml=True)
print(res1.summary())

# ------------------------------
# 10) Step 2: Random Intercept + Random Slope (IT_within)
# ------------------------------
m2 = smf.mixedlm(
    f"{Y} ~ {rhs}",
    data=d,
    groups=d[ID],
    re_formula="1 + IT_within"
)

try:
    res2 = m2.fit(method="lbfgs", reml=True)
except Exception as e1:
    print("LBFGS failed, trying Powell. Error:", e1)
    res2 = m2.fit(method="powell", reml=True)

print(res2.summary())

print("\nModel Comparison:")
print(f"  Log-Likelihood (RI)    : {res1.llf:.2f}")
print(f"  Log-Likelihood (RI+RS) : {res2.llf:.2f}")

# ===========================================================
# ✅ Nakagawa R² (prediction-based, random slope 안전 버전)
# ===========================================================

def mixedlm_r2_pred(res):
    """
    Prediction-based Nakagawa R²
    Works for random intercept + random slope models
    """
    m = res.model

    # Fixed effect fitted values (Xβ)
    yhat_fix = m.exog @ res.fe_params.values

    # Full fitted values (Xβ + Zu)
    yhat_full = np.asarray(res.fittedvalues)

    # Random effect contribution (Zu)
    Zu = yhat_full - yhat_fix

    # Variance components
    var_fix   = np.var(yhat_fix, ddof=1)
    var_re    = np.var(Zu, ddof=1)
    var_resid = res.scale

    denom = var_fix + var_re + var_resid

    R2_marginal   = var_fix / denom
    R2_conditional = (var_fix + var_re) / denom

    return {
        "R2_marginal": R2_marginal,
        "R2_conditional": R2_conditional,
        "var_fixed": var_fix,
        "var_random": var_re,
        "var_residual": var_resid
    }

print("\n--- R² (Nakagawa, prediction-based) ---")

r2_res1 = mixedlm_r2_pred(res1)
print("\nRandom Intercept Model:")
for k, v in r2_res1.items():
    print(f"{k}: {v:.4f}")

r2_res2 = mixedlm_r2_pred(res2)
print("\nRandom Intercept + Random Slope Model:")
for k, v in r2_res2.items():
    print(f"{k}: {v:.4f}")

PCA weights (events, damage): [[0.70710678 0.70710678]]
Explained variance ratio: [0.912226]
                     Mixed Linear Model Regression Results
Model:                     MixedLM        Dependent Variable:        RevPar_log
No. Observations:          70867          Method:                    REML      
No. Groups:                10228          Scale:                     0.0162    
Min. group size:           1              Log-Likelihood:            30032.4067
Max. group size:           18             Converged:                 Yes       
Mean group size:           6.9                                                 
-------------------------------------------------------------------------------
                                   Coef.  Std.Err.    z     P>|z| [0.025 0.975]
-------------------------------------------------------------------------------
Intercept                           4.508    0.012  383.739 0.000  4.485  4.531
C(ChainSegment)[T.Luxury]           0.070    0.0

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)


                     Mixed Linear Model Regression Results
Model:                     MixedLM        Dependent Variable:        RevPar_log
No. Observations:          70867          Method:                    REML      
No. Groups:                10228          Scale:                     0.0141    
Min. group size:           1              Log-Likelihood:            31310.5614
Max. group size:           18             Converged:                 Yes       
Mean group size:           6.9                                                 
-------------------------------------------------------------------------------
                                   Coef.  Std.Err.    z     P>|z| [0.025 0.975]
-------------------------------------------------------------------------------
Intercept                           4.515    0.012  388.856 0.000  4.492  4.538
C(ChainSegment)[T.Luxury]           0.065    0.012    5.325 0.000  0.041  0.089
C(ChainSegment)[T.Midscale]         0.017    0.011    1.558 0

In [45]:
#3 IT variables - PCA risk, Tropical

# ------------------------------
# Utility: prediction-based Nakagawa R²
# ------------------------------
def mixedlm_r2_pred(res):
    m = res.model
    yhat_fix = m.exog @ res.fe_params.values
    yhat_full = np.asarray(res.fittedvalues)
    Zu = yhat_full - yhat_fix

    var_fix = np.var(yhat_fix, ddof=1)
    var_re  = np.var(Zu, ddof=1)
    var_res = res.scale

    denom = var_fix + var_re + var_res
    return {
        "R2_marginal": var_fix / denom,
        "R2_conditional": (var_fix + var_re) / denom,
        "var_fixed": var_fix,
        "var_random": var_re,
        "var_residual": var_res,
    }

# ------------------------------
# Utility: AR(1) by group
# ------------------------------
def ar1_by_property(data, id_col, year_col, resid):
    tmp = data[[id_col, year_col]].copy()
    tmp["resid"] = resid

    def ar1_group(g):
        g = g.sort_values(year_col)
        if len(g) > 2:
            return g["resid"].autocorr(lag=1)
        return np.nan

    return tmp.groupby(id_col).apply(ar1_group)

# ===========================================================
# 0) Safe: reset_index는 drop=True (중복 level_0 방지)
# ===========================================================
df = df.copy()
df = df.reset_index(drop=True)

# ===========================================================
# 1) 변수 설정 (너 환경에 맞춰 이미 정의돼있다고 가정)
# ===========================================================
# 필수: 아래 변수들은 이미 너 노트북에 정의돼 있어야 함
# Y, ID, REGION, SEG, YEAR, CONTROLS_term
# 예시:
# Y="RevPar_log"; ID="PropertyID"; REGION="noaa_region9"; SEG="ChainSegment"; YEAR="Year"

# IT log 변수
IT_LAB = "ITTotalLabor_log"
IT_SER = "ITService_log"
IT_SYS = "ITSystem_log"

# PCA risk source
EV_LOG  = "Tropical_events_log"
DMG_LOG = "Tropical_damage_log"

# -----------------------------------------------------------
# 2) 분석 샘플 구축 (PCA risk 포함)
# -----------------------------------------------------------
base_cols = [Y, ID, REGION, SEG, YEAR, IT_LAB, IT_SER, IT_SYS, EV_LOG, DMG_LOG] + CONTROLS_term
d2 = df[base_cols].dropna().copy()

d2[YEAR] = pd.to_numeric(d2[YEAR], errors="coerce")
d2 = d2.dropna(subset=[YEAR])
d2[YEAR] = d2[YEAR].astype(int)

d2[SEG] = d2[SEG].astype("category")
d2[REGION] = d2[REGION].astype("category")

# -----------------------------------------------------------
# 3) PCA Risk 생성 (표준화 후 PCA + 부호 고정)
# -----------------------------------------------------------
X = d2[[EV_LOG, DMG_LOG]].values
X_std = StandardScaler(with_mean=True, with_std=True).fit_transform(X)

pca = PCA(n_components=1, random_state=0)
risk_pca = pca.fit_transform(X_std).ravel()

if np.corrcoef(risk_pca, d2[EV_LOG].values)[0, 1] < 0:
    risk_pca = -risk_pca
    pca.components_ = -pca.components_

d2["Risk_pca"] = risk_pca
d2["Risk_z"] = (d2["Risk_pca"] - d2["Risk_pca"].mean()) / d2["Risk_pca"].std(ddof=0)

print("PCA weights (events, damage):", pca.components_)
print("Explained variance ratio:", pca.explained_variance_ratio_)

# -----------------------------------------------------------
# 4) IT within/between 분해 + z-score
# -----------------------------------------------------------
def decompose_it(data, var, group=ID):
    data[f"{var}_mean_hotel"] = data.groupby(group)[var].transform("mean")
    data[f"{var}_within"] = data[var] - data[f"{var}_mean_hotel"]
    for col in [f"{var}_mean_hotel", f"{var}_within"]:
        sd = data[col].std(ddof=0)
        if sd > 0:
            data[col] = (data[col] - data[col].mean()) / sd
    return data

for it_var in [IT_LAB, IT_SER, IT_SYS]:
    d2 = decompose_it(d2, it_var)

# interactions
for it_var in [IT_LAB, IT_SER, IT_SYS]:
    d2[f"{it_var}_w_x_R"] = d2[f"{it_var}_within"] * d2["Risk_z"]
    d2[f"{it_var}_b_x_R"] = d2[f"{it_var}_mean_hotel"] * d2["Risk_z"]

# controls standardize (age 제외를 원하면 여기서 처리)
for col in [c for c in CONTROLS_term if c != "age"]:
    if col in d2.columns:
        sd = d2[col].std(ddof=0)
        if sd > 0:
            d2[col] = (d2[col] - d2[col].mean()) / sd

# -----------------------------------------------------------
# 5) Formula 구성
# -----------------------------------------------------------
rhs_terms = ["Risk_z"]
for it_var in [IT_LAB, IT_SER, IT_SYS]:
    rhs_terms += [
        f"{it_var}_within",
        f"{it_var}_mean_hotel",
        f"{it_var}_w_x_R",
        f"{it_var}_b_x_R",
    ]
rhs_terms += CONTROLS_term + [f"C({SEG})", f"C({REGION})", f"C({YEAR})"]
rhs = " + ".join(rhs_terms)
formula = f"{Y} ~ {rhs}"

# ===========================================================
# 6) 모델 적합: (A) REML for inference (B) ML for comparison
# ===========================================================

def fit_models(reml=True):
    # Model 1: Random Intercept
    m1 = smf.mixedlm(formula, data=d2, groups=d2[ID], re_formula="1")
    res1 = m1.fit(method="lbfgs", reml=reml)

    # Model 2: Random Intercept + Random Slope (IT labor within)
    slope_var = f"{IT_LAB}_within"
    m2 = smf.mixedlm(formula, data=d2, groups=d2[ID], re_formula=f"1 + {slope_var}")
    try:
        res2 = m2.fit(method="lbfgs", reml=reml)
    except Exception:
        res2 = m2.fit(method="powell", reml=reml)

    return res1, res2

# (A) REML results (계수 해석용)
res1_reml, res2_reml = fit_models(reml=True)
print("\n===== REML: Model 1 (RI) =====")
print(res1_reml.summary())
print("\n===== REML: Model 2 (RI+RS) =====")
print(res2_reml.summary())

# (B) ML results (모델 비교용)
res1_ml, res2_ml = fit_models(reml=False)

print("\n===== MODEL COMPARISON (ML) =====")
print(f"LogLik RI     : {res1_ml.llf:.2f}")
print(f"LogLik RI+RS  : {res2_ml.llf:.2f}")
print(f"AIC   RI      : {res1_ml.aic:.2f}")
print(f"AIC   RI+RS   : {res2_ml.aic:.2f}")

# Likelihood Ratio Test (approx)
lr = 2 * (res2_ml.llf - res1_ml.llf)
df_diff = len(res2_ml.params) - len(res1_ml.params)
p_lr = chi2.sf(lr, df=df_diff) if df_diff > 0 else np.nan
print(f"LRT statistic : {lr:.2f} (df={df_diff}), p={p_lr:.4g}")

# ===========================================================
# 7) Diagnostics: R² / AR(1) / Ljung-Box (REML 기준)
# ===========================================================
def diagnostics(label, res):
    print(f"\n--- Diagnostics: {label} ---")

    # R²
    r2 = mixedlm_r2_pred(res)
    print("R² (Nakagawa, pred-based):")
    print({k: round(v, 4) for k, v in r2.items()})

    # AR(1)
    ar1 = ar1_by_property(d2, ID, YEAR, res.resid)
    print("\nAR(1) by Property:")
    print(ar1.describe())

    # Ljung–Box (pooled)
    print("\nLjung–Box (pooled residuals):")
    lb = acorr_ljungbox(pd.Series(res.resid).dropna(), lags=[1,2,3,4], return_df=True)
    print(lb)

diagnostics("Model 1 (RI, REML)", res1_reml)
diagnostics("Model 2 (RI+RS, REML)", res2_reml)

PCA weights (events, damage): [[0.70710678 0.70710678]]
Explained variance ratio: [0.96580768]


/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)



===== REML: Model 1 (RI) =====
                    Mixed Linear Model Regression Results
Model:                    MixedLM        Dependent Variable:        RevPar_log
No. Observations:         4695           Method:                    REML      
No. Groups:               2262           Scale:                     0.0183    
Min. group size:          1              Log-Likelihood:            54.9137   
Max. group size:          10             Converged:                 Yes       
Mean group size:          2.1                                                 
------------------------------------------------------------------------------
                                   Coef.  Std.Err.    z    P>|z| [0.025 0.975]
------------------------------------------------------------------------------
Intercept                           4.402    0.050  87.938 0.000  4.304  4.500
C(ChainSegment)[T.Luxury]           0.447    0.051   8.773 0.000  0.347  0.547
C(ChainSegment)[T.Midscale]         0.407

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)



===== MODEL COMPARISON (ML) =====
LogLik RI     : 228.27
LogLik RI+RS  : 249.00
AIC   RI      : -360.54
AIC   RI+RS   : -397.99
LRT statistic : 41.45 (df=2), p=9.972e-10

--- Diagnostics: Model 1 (RI, REML) ---
R² (Nakagawa, pred-based):
{'R2_marginal': np.float64(0.8038), 'R2_conditional': np.float64(0.9555), 'var_fixed': np.float64(0.3301), 'var_random': np.float64(0.0623), 'var_residual': np.float64(0.0183)}

AR(1) by Property:
count    550.000000
mean       0.054748
std        0.710077
min       -1.000000
25%       -0.523360
50%        0.111263
75%        0.755438
max        1.000000
dtype: float64

Ljung–Box (pooled residuals):
      lb_stat     lb_pvalue
1    8.846065  2.937215e-03
2  130.527331  4.532629e-29
3  201.388193  2.114498e-43
4  221.244727  1.011741e-46

--- Diagnostics: Model 2 (RI+RS, REML) ---
R² (Nakagawa, pred-based):
{'R2_marginal': np.float64(0.8031), 'R2_conditional': np.float64(0.962), 'var_fixed': np.float64(0.3288), 'var_random': np.float64(0.0651), 'var_re

/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1334/554325656.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return tmp.groupby(id_col).apply(ar1_group)
/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1334/554325656.py:44: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return tmp.groupby(id_col).apply(ar1_group)


In [47]:
#3 IT variables - PCA risk, FIRE

# ------------------------------
# Utility: prediction-based Nakagawa R²
# ------------------------------
def mixedlm_r2_pred(res):
    m = res.model
    yhat_fix = m.exog @ res.fe_params.values
    yhat_full = np.asarray(res.fittedvalues)
    Zu = yhat_full - yhat_fix

    var_fix = np.var(yhat_fix, ddof=1)
    var_re  = np.var(Zu, ddof=1)
    var_res = res.scale

    denom = var_fix + var_re + var_res
    return {
        "R2_marginal": var_fix / denom,
        "R2_conditional": (var_fix + var_re) / denom,
        "var_fixed": var_fix,
        "var_random": var_re,
        "var_residual": var_res,
    }

# ------------------------------
# Utility: AR(1) by group
# ------------------------------
def ar1_by_property(data, id_col, year_col, resid):
    tmp = data[[id_col, year_col]].copy()
    tmp["resid"] = resid

    def ar1_group(g):
        g = g.sort_values(year_col)
        if len(g) > 2:
            return g["resid"].autocorr(lag=1)
        return np.nan

    return tmp.groupby(id_col).apply(ar1_group)

# ===========================================================
# 0) Safe: reset_index는 drop=True (중복 level_0 방지)
# ===========================================================
df = df.copy()
df = df.reset_index(drop=True)

# ===========================================================
# 1) 변수 설정 (너 환경에 맞춰 이미 정의돼있다고 가정)
# ===========================================================
# 필수: 아래 변수들은 이미 너 노트북에 정의돼 있어야 함
# Y, ID, REGION, SEG, YEAR, CONTROLS_term
# 예시:
# Y="RevPar_log"; ID="PropertyID"; REGION="noaa_region9"; SEG="ChainSegment"; YEAR="Year"

# IT log 변수
IT_LAB = "ITTotalLabor_log"
IT_SER = "ITService_log"
IT_SYS = "ITSystem_log"

# PCA risk source
EV_LOG  = "Fire_events_log"
DMG_LOG = "Fire_damage_log"

# -----------------------------------------------------------
# 2) 분석 샘플 구축 (PCA risk 포함)
# -----------------------------------------------------------
base_cols = [Y, ID, REGION, SEG, YEAR, IT_LAB, IT_SER, IT_SYS, EV_LOG, DMG_LOG] + CONTROLS_term
d2 = df[base_cols].dropna().copy()

d2[YEAR] = pd.to_numeric(d2[YEAR], errors="coerce")
d2 = d2.dropna(subset=[YEAR])
d2[YEAR] = d2[YEAR].astype(int)

d2[SEG] = d2[SEG].astype("category")
d2[REGION] = d2[REGION].astype("category")

# -----------------------------------------------------------
# 3) PCA Risk 생성 (표준화 후 PCA + 부호 고정)
# -----------------------------------------------------------
X = d2[[EV_LOG, DMG_LOG]].values
X_std = StandardScaler(with_mean=True, with_std=True).fit_transform(X)

pca = PCA(n_components=1, random_state=0)
risk_pca = pca.fit_transform(X_std).ravel()

if np.corrcoef(risk_pca, d2[EV_LOG].values)[0, 1] < 0:
    risk_pca = -risk_pca
    pca.components_ = -pca.components_

d2["Risk_pca"] = risk_pca
d2["Risk_z"] = (d2["Risk_pca"] - d2["Risk_pca"].mean()) / d2["Risk_pca"].std(ddof=0)

print("PCA weights (events, damage):", pca.components_)
print("Explained variance ratio:", pca.explained_variance_ratio_)

# -----------------------------------------------------------
# 4) IT within/between 분해 + z-score
# -----------------------------------------------------------
def decompose_it(data, var, group=ID):
    data[f"{var}_mean_hotel"] = data.groupby(group)[var].transform("mean")
    data[f"{var}_within"] = data[var] - data[f"{var}_mean_hotel"]
    for col in [f"{var}_mean_hotel", f"{var}_within"]:
        sd = data[col].std(ddof=0)
        if sd > 0:
            data[col] = (data[col] - data[col].mean()) / sd
    return data

for it_var in [IT_LAB, IT_SER, IT_SYS]:
    d2 = decompose_it(d2, it_var)

# interactions
for it_var in [IT_LAB, IT_SER, IT_SYS]:
    d2[f"{it_var}_w_x_R"] = d2[f"{it_var}_within"] * d2["Risk_z"]
    d2[f"{it_var}_b_x_R"] = d2[f"{it_var}_mean_hotel"] * d2["Risk_z"]

# controls standardize (age 제외를 원하면 여기서 처리)
for col in [c for c in CONTROLS_term if c != "age"]:
    if col in d2.columns:
        sd = d2[col].std(ddof=0)
        if sd > 0:
            d2[col] = (d2[col] - d2[col].mean()) / sd

# -----------------------------------------------------------
# 5) Formula 구성
# -----------------------------------------------------------
rhs_terms = ["Risk_z"]
for it_var in [IT_LAB, IT_SER, IT_SYS]:
    rhs_terms += [
        f"{it_var}_within",
        f"{it_var}_mean_hotel",
        f"{it_var}_w_x_R",
        f"{it_var}_b_x_R",
    ]
rhs_terms += CONTROLS_term + [f"C({SEG})", f"C({REGION})", f"C({YEAR})"]
rhs = " + ".join(rhs_terms)
formula = f"{Y} ~ {rhs}"

# ===========================================================
# 6) 모델 적합: (A) REML for inference (B) ML for comparison
# ===========================================================

def fit_models(reml=True):
    # Model 1: Random Intercept
    m1 = smf.mixedlm(formula, data=d2, groups=d2[ID], re_formula="1")
    res1 = m1.fit(method="lbfgs", reml=reml)

    # Model 2: Random Intercept + Random Slope (IT labor within)
    slope_var = f"{IT_LAB}_within"
    m2 = smf.mixedlm(formula, data=d2, groups=d2[ID], re_formula=f"1 + {slope_var}")
    try:
        res2 = m2.fit(method="lbfgs", reml=reml)
    except Exception:
        res2 = m2.fit(method="powell", reml=reml)

    return res1, res2

# (A) REML results (계수 해석용)
res1_reml, res2_reml = fit_models(reml=True)
print("\n===== REML: Model 1 (RI) =====")
print(res1_reml.summary())
print("\n===== REML: Model 2 (RI+RS) =====")
print(res2_reml.summary())

# (B) ML results (모델 비교용)
res1_ml, res2_ml = fit_models(reml=False)

print("\n===== MODEL COMPARISON (ML) =====")
print(f"LogLik RI     : {res1_ml.llf:.2f}")
print(f"LogLik RI+RS  : {res2_ml.llf:.2f}")
print(f"AIC   RI      : {res1_ml.aic:.2f}")
print(f"AIC   RI+RS   : {res2_ml.aic:.2f}")

# Likelihood Ratio Test (approx)
lr = 2 * (res2_ml.llf - res1_ml.llf)
df_diff = len(res2_ml.params) - len(res1_ml.params)
p_lr = chi2.sf(lr, df=df_diff) if df_diff > 0 else np.nan
print(f"LRT statistic : {lr:.2f} (df={df_diff}), p={p_lr:.4g}")

# ===========================================================
# 7) Diagnostics: R² / AR(1) / Ljung-Box (REML 기준)
# ===========================================================
def diagnostics(label, res):
    print(f"\n--- Diagnostics: {label} ---")

    # R²
    r2 = mixedlm_r2_pred(res)
    print("R² (Nakagawa, pred-based):")
    print({k: round(v, 4) for k, v in r2.items()})

    # AR(1)
    ar1 = ar1_by_property(d2, ID, YEAR, res.resid)
    print("\nAR(1) by Property:")
    print(ar1.describe())

    # Ljung–Box (pooled)
    print("\nLjung–Box (pooled residuals):")
    lb = acorr_ljungbox(pd.Series(res.resid).dropna(), lags=[1,2,3,4], return_df=True)
    print(lb)

diagnostics("Model 1 (RI, REML)", res1_reml)
diagnostics("Model 2 (RI+RS, REML)", res2_reml)

PCA weights (events, damage): [[0.70710678 0.70710678]]
Explained variance ratio: [0.91134629]


/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)



===== REML: Model 1 (RI) =====
                    Mixed Linear Model Regression Results
Model:                    MixedLM        Dependent Variable:        RevPar_log
No. Observations:         4695           Method:                    REML      
No. Groups:               2262           Scale:                     0.0182    
Min. group size:          1              Log-Likelihood:            60.9630   
Max. group size:          10             Converged:                 Yes       
Mean group size:          2.1                                                 
------------------------------------------------------------------------------
                                   Coef.  Std.Err.    z    P>|z| [0.025 0.975]
------------------------------------------------------------------------------
Intercept                           4.408    0.050  88.651 0.000  4.311  4.506
C(ChainSegment)[T.Luxury]           0.458    0.051   9.012 0.000  0.358  0.557
C(ChainSegment)[T.Midscale]         0.434

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)



===== MODEL COMPARISON (ML) =====
LogLik RI     : 234.11
LogLik RI+RS  : 253.68
AIC   RI      : -372.22
AIC   RI+RS   : -407.36
LRT statistic : 39.15 (df=2), p=3.16e-09

--- Diagnostics: Model 1 (RI, REML) ---
R² (Nakagawa, pred-based):
{'R2_marginal': np.float64(0.8047), 'R2_conditional': np.float64(0.9557), 'var_fixed': np.float64(0.3308), 'var_random': np.float64(0.0621), 'var_residual': np.float64(0.0182)}

AR(1) by Property:
count    550.000000
mean       0.072547
std        0.705172
min       -1.000000
25%       -0.504890
50%        0.139762
75%        0.734669
max        1.000000
dtype: float64

Ljung–Box (pooled residuals):
      lb_stat     lb_pvalue
1    8.096283  4.435613e-03
2  131.336842  3.023897e-29
3  204.238297  5.120790e-44
4  226.744844  6.626803e-48

--- Diagnostics: Model 2 (RI+RS, REML) ---
R² (Nakagawa, pred-based):
{'R2_marginal': np.float64(0.8039), 'R2_conditional': np.float64(0.9623), 'var_fixed': np.float64(0.3292), 'var_random': np.float64(0.0649), 'var_re

/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1334/365632791.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return tmp.groupby(id_col).apply(ar1_group)
/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1334/365632791.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return tmp.groupby(id_col).apply(ar1_group)


In [ ]:
# 1️⃣ log return 생성
df["RevPar_return"] = df.groupby(ID)["RevPar_log"].diff()

# 2️⃣ 5년 rolling std
df["RevPar_vol_5y"] = (
    df.groupby(ID)["RevPar_return"]
      .rolling(window=5, min_periods=3)
      .std()
      .reset_index(level=0, drop=True)
)

In [47]:
#Robustness check

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import acorr_ljungbox

try:
    from linearmodels.panel import PanelOLS
    PANEL_OK = True
except ImportError:
    PANEL_OK = False
    print("linearmodels is not installed. Panel FE robustness will be skipped.")

# -------------------------------------------------
# BASIC SETTINGS
# -------------------------------------------------
df = df.reset_index(drop=True)

ID     = "PropertyID"
REGION = "noaa_region9"
SEG    = "ChainSegment"

if "Year of Data" in df.columns and "Year" not in df.columns:
    df = df.rename(columns={"Year of Data": "Year"})
YEAR = "Year"

Y = "RevPar_log"

CONTROLS = [
    "TotalLabor", "MaintTotalExp", "UtilTotalExp",
    "Insurance", "SandMTotalExpense", "MktingTotalExp",
    "TotalMgmtFee", "PropertyTax", "Rooms", "age"
]

money_vars = [
    "ITTotalExpense", "ITTotalLabor", "ITService", "ITSystem",
    "TotalLabor", "MaintTotalExp", "UtilTotalExp",
    "Insurance", "SandMTotalExpense", "MktingTotalExp",
    "TotalMgmtFee", "PropertyTax", "Rooms"
]

def safe_transform(series, use_asinh_if_negative=True):
    s = pd.to_numeric(series, errors="coerce")
    if use_asinh_if_negative and (s.dropna() < 0).any():
        return np.arcsinh(s)
    else:
        return np.log1p(s.clip(lower=0))

def zscore_safe(series):
    s = pd.to_numeric(series, errors="coerce")
    sd = s.std(skipna=True, ddof=0)
    if pd.notnull(sd) and sd > 0:
        return (s - s.mean()) / sd
    return s

# transformed vars
for col in money_vars:
    if col in df.columns and f"tr_{col}" not in df.columns:
        df[f"tr_{col}"] = safe_transform(df[col], use_asinh_if_negative=True)

CONTROLS_term = []
for c in CONTROLS:
    if c == "age":
        CONTROLS_term.append("age")
    else:
        CONTROLS_term.append(f"tr_{c}")

def make_pca_risk(df, event_col, damage_col, out_col):
    tmp = df[[event_col, damage_col]].dropna().copy()
    X_std = StandardScaler().fit_transform(tmp[[event_col, damage_col]].values)

    pca = PCA(n_components=1, random_state=0)
    pc1 = pca.fit_transform(X_std).ravel()

    if np.corrcoef(pc1, tmp[event_col].values)[0, 1] < 0:
        pc1 = -pc1
        pca.components_ = -pca.components_

    df[out_col] = np.nan
    df.loc[tmp.index, out_col] = pc1
    df[out_col] = zscore_safe(df[out_col])

    print(f"\n{out_col}")
    print("PCA weights:", pca.components_)
    print("Explained variance ratio:", pca.explained_variance_ratio_)
    return df

if "Tropical_events_log" in df.columns and "Tropical_damage_log" in df.columns and "Tropical_Risk_z" not in df.columns:
    df = make_pca_risk(df, "Tropical_events_log", "Tropical_damage_log", "Tropical_Risk_z")

if "Fire_events_log" in df.columns and "Fire_damage_log" in df.columns and "Fire_Risk_z" not in df.columns:
    df = make_pca_risk(df, "Fire_events_log", "Fire_damage_log", "Fire_Risk_z")

def prepare_panel(df):
    d = df.copy()

    # defensive rename
    if "Year" not in d.columns and "Year of Data" in d.columns:
        d = d.rename(columns={"Year of Data": "Year"})

    if "Year" not in d.columns:
        raise KeyError(
            f"'Year' column not found. Available columns similar to year: "
            f"{[c for c in d.columns if 'year' in c.lower()]}"
        )

    d["Year"] = pd.to_numeric(d["Year"], errors="coerce")
    d = d.dropna(subset=[ID, "Year"])
    d["Year"] = d["Year"].astype(int)
    return d

def add_lagged_structure(df, it_col, risk_col, lag=1, standardize=True):
    d = df.copy().sort_values([ID, YEAR])

    lag_col = f"lag_{it_col}"
    d[lag_col] = d.groupby(ID)[it_col].shift(lag)

    mean_col = f"{lag_col}_mean_hotel"
    d[mean_col] = d.groupby(ID)[lag_col].transform("mean")

    within_col = f"{lag_col}_within"
    d[within_col] = d[lag_col] - d[mean_col]

    risk_mean_col = f"{risk_col}_mean_hotel"
    risk_within_col = f"{risk_col}_within_hotel"
    d[risk_mean_col] = d.groupby(ID)[risk_col].transform("mean")
    d[risk_within_col] = d[risk_col] - d[risk_mean_col]

    if standardize:
        for c in [lag_col, mean_col, within_col, risk_col, risk_mean_col, risk_within_col]:
            if c in d.columns:
                d[c] = zscore_safe(d[c])

    w_int = f"{lag_col}_w_x_R"
    b_int = f"{lag_col}_b_x_Rw"
    d[w_int] = d[within_col] * d[risk_col]
    d[b_int] = d[mean_col] * d[risk_within_col]

    names = {
        "lag_col": lag_col,
        "mean_col": mean_col,
        "within_col": within_col,
        "risk_mean_col": risk_mean_col,
        "risk_within_col": risk_within_col,
        "w_int": w_int,
        "b_int": b_int
    }
    return d, names

def fit_mixedlm_model(df, it_col, risk_col, exclude_2020=False, random_slope=True):
    d = prepare_panel(df)
    if exclude_2020:
        d = d[d[YEAR] != 2020].copy()

    d, names = add_lagged_structure(d, it_col, risk_col, lag=1, standardize=True)

    needed = [
        Y, ID, REGION, SEG, YEAR, risk_col,
        names["mean_col"], names["within_col"],
        names["risk_within_col"], names["w_int"], names["b_int"]
    ] + CONTROLS_term
    d = d.dropna(subset=needed).copy()

    d[SEG] = d[SEG].astype("category")
    d[REGION] = d[REGION].astype("category")
    d[YEAR] = d[YEAR].astype("category")

    rhs = " + ".join(
        [risk_col, names["within_col"], names["mean_col"], names["w_int"], names["b_int"]]
        + CONTROLS_term
        + [f"C({SEG})", f"C({REGION})", f"C({YEAR})"]
    )

    re_formula = "1"
    if random_slope:
        re_formula = f"1 + {names['within_col']}"

    model = smf.mixedlm(f"{Y} ~ {rhs}", data=d, groups=d[ID], re_formula=re_formula)

    try:
        res = model.fit(method="lbfgs", reml=True)
    except Exception:
        res = model.fit(method="powell", reml=True)

    return res, d, names

def fit_panel_fe_cluster(df, it_col, risk_col, exclude_2020=False):
    if not PANEL_OK:
        return None, None, None

    d = prepare_panel(df)
    if exclude_2020:
        d = d[d[YEAR] != 2020].copy()

    d, names = add_lagged_structure(d, it_col, risk_col, lag=1, standardize=True)

    needed = [
        Y, ID, YEAR, risk_col,
        names["mean_col"], names["within_col"],
        names["risk_within_col"], names["w_int"], names["b_int"]
    ] + CONTROLS_term
    d = d.dropna(subset=needed).copy()

    d = pd.get_dummies(d, columns=[REGION, SEG], drop_first=True)
    exog_cols = [risk_col, names["within_col"], names["mean_col"], names["w_int"], names["b_int"]] + CONTROLS_term
    exog_cols += [c for c in d.columns if c.startswith(REGION + "_") or c.startswith(SEG + "_")]

    d = d.set_index([ID, YEAR])

    mod = PanelOLS(
        dependent=d[Y],
        exog=d[exog_cols],
        entity_effects=True,
        time_effects=True,
        drop_absorbed=True
    )
    res = mod.fit(cov_type="clustered", cluster_entity=True)
    return res, d, names

def extract_full_mixed_results(res):
    return pd.DataFrame({
        "term": res.params.index,
        "coef": res.params.values,
        "se": res.bse.values,
        "p": res.pvalues.values
    })

def extract_full_panel_results(res):
    if res is None:
        return pd.DataFrame()
    return pd.DataFrame({
        "term": res.params.index,
        "coef": res.params.values,
        "se": res.std_errors.values,
        "p": res.pvalues.values
    })

def extract_key_terms_mixed(res, names, risk_col, label):
    terms = [risk_col, names["within_col"], names["mean_col"], names["w_int"], names["b_int"]]
    out = []
    for t in terms:
        if t in res.params.index:
            out.append({
                "model": label,
                "term": t,
                "coef": res.params[t],
                "se": res.bse[t],
                "p": res.pvalues[t]
            })
    return pd.DataFrame(out)

def extract_key_terms_panel(res, names, risk_col, label):
    if res is None:
        return pd.DataFrame()
    terms = [risk_col, names["within_col"], names["mean_col"], names["w_int"], names["b_int"]]
    out = []
    for t in terms:
        if t in res.params.index:
            out.append({
                "model": label,
                "term": t,
                "coef": res.params[t],
                "se": res.std_errors[t],
                "p": res.pvalues[t]
            })
    return pd.DataFrame(out)

def pretty_compare(df_coef):
    if df_coef.empty:
        return df_coef
    tmp = df_coef.copy()
    tmp["coef_se"] = tmp.apply(lambda x: f"{x['coef']:.3f} ({x['se']:.3f}), p={x['p']:.3f}", axis=1)
    return tmp.pivot(index="term", columns="model", values="coef_se")

def compute_vif_from_mixedlm_result(res):
    X = pd.DataFrame(res.model.exog, columns=res.model.exog_names)
    rows = []
    for i, col in enumerate(X.columns):
        try:
            vif = variance_inflation_factor(X.values, i)
        except Exception:
            vif = np.nan
        rows.append({"variable": col, "VIF": vif})
    return pd.DataFrame(rows).sort_values("VIF", ascending=False)

def mixedlm_r2_pred(res):
    m = res.model
    yhat_fix = m.exog @ res.fe_params.values
    yhat_full = np.asarray(res.fittedvalues)
    Zu = yhat_full - yhat_fix

    var_fix = np.var(yhat_fix, ddof=1)
    var_re = np.var(Zu, ddof=1)
    var_resid = res.scale
    denom = var_fix + var_re + var_resid

    return pd.DataFrame([{
        "R2_marginal": var_fix / denom,
        "R2_conditional": (var_fix + var_re) / denom,
        "var_fixed": var_fix,
        "var_random": var_re,
        "var_residual": var_resid
    }])

def residual_autocorr_diagnostics(res, data_used):
    dtmp = data_used.copy()
    dtmp["_resid"] = res.resid

    def ar1_group(g):
        g = g.sort_values(YEAR)
        if len(g) > 2:
            return g["_resid"].autocorr(lag=1)
        return np.nan

    ar1_vals = dtmp.groupby(ID).apply(ar1_group).dropna()
    ar1_desc = ar1_vals.describe().to_frame("AR1")

    lb = acorr_ljungbox(dtmp["_resid"].dropna(), lags=[1,2,3,4], return_df=True)
    return ar1_desc, lb

def panel_r2_table(res):
    if res is None:
        return pd.DataFrame()
    return pd.DataFrame([{
        "R2_within": res.rsquared_within,
        "R2_between": res.rsquared_between,
        "R2_overall": res.rsquared_overall
    }])

def build_full_comparison_table(mixed_full_res, mixed_no20_res, panel_res):
    """
    모든 계수를 wide table로 정리
    """
    mf = extract_full_mixed_results(mixed_full_res).copy()
    mf["model"] = "MixedLM_full"

    mn = extract_full_mixed_results(mixed_no20_res).copy()
    mn["model"] = "MixedLM_no2020"

    if panel_res is not None:
        pf = extract_full_panel_results(panel_res).copy()
        pf["model"] = "PanelFE_cluster"
        full_long = pd.concat([mf, mn, pf], ignore_index=True)
    else:
        full_long = pd.concat([mf, mn], ignore_index=True)

    full_long["coef_se_p"] = full_long.apply(
        lambda x: f"{x['coef']:.3f} ({x['se']:.3f}), p={x['p']:.3f}",
        axis=1
    )

    full_wide = full_long.pivot_table(
        index="term",
        columns="model",
        values="coef_se_p",
        aggfunc="first"
    )

    return full_long, full_wide

def build_fit_statistics_table(mixed_full_res, mixed_no20_res, panel_res=None):
    rows = []

    rows.append({
        "model": "MixedLM_full",
        "nobs": getattr(mixed_full_res, "nobs", np.nan),
        "llf": getattr(mixed_full_res, "llf", np.nan),
        "aic": getattr(mixed_full_res, "aic", np.nan),
        "bic": getattr(mixed_full_res, "bic", np.nan)
    })

    rows.append({
        "model": "MixedLM_no2020",
        "nobs": getattr(mixed_no20_res, "nobs", np.nan),
        "llf": getattr(mixed_no20_res, "llf", np.nan),
        "aic": getattr(mixed_no20_res, "aic", np.nan),
        "bic": getattr(mixed_no20_res, "bic", np.nan)
    })

    if panel_res is not None:
        rows.append({
            "model": "PanelFE_cluster",
            "nobs": getattr(panel_res, "nobs", np.nan),
            "llf": np.nan,
            "aic": np.nan,
            "bic": np.nan
        })

    return pd.DataFrame(rows)

def fit_mixedlm_model_ldv(df, it_col, risk_col, exclude_2020=False, random_slope=True):

    d = prepare_panel(df)

    if exclude_2020:
        d = d[d[YEAR] != 2020].copy()

    # lagged dependent variable
    d = d.sort_values([ID, YEAR])
    d["lag_RevPar_log"] = d.groupby(ID)[Y].shift(1)

    d, names = add_lagged_structure(
        d,
        it_col,
        risk_col,
        lag=1,
        standardize=True
    )

    needed = [
        Y,
        "lag_RevPar_log",
        ID,
        REGION,
        SEG,
        YEAR,
        risk_col,
        names["mean_col"],
        names["within_col"],
        names["risk_within_col"],
        names["w_int"],
        names["b_int"]
    ] + CONTROLS_term

    d = d.dropna(subset=needed).copy()

    d[SEG] = d[SEG].astype("category")
    d[REGION] = d[REGION].astype("category")
    d[YEAR] = d[YEAR].astype("category")

    rhs = " + ".join(
        [
            "lag_RevPar_log",
            risk_col,
            names["within_col"],
            names["mean_col"],
            names["w_int"],
            names["b_int"]
        ]
        + CONTROLS_term
        + [
            f"C({SEG})",
            f"C({REGION})",
            f"C({YEAR})"
        ]
    )

    re_formula = "1"

    if random_slope:
        re_formula = f"1 + {names['within_col']}"

    model = smf.mixedlm(
        f"{Y} ~ {rhs}",
        data=d,
        groups=d[ID],
        re_formula=re_formula
    )

    try:
        res = model.fit(method="lbfgs", reml=True)
    except Exception:
        res = model.fit(method="powell", reml=True)

    return res, d, names


def fit_panel_fe_cluster_ldv(df, it_col, risk_col, exclude_2020=False):
    if not PANEL_OK:
        return None, None, None

    # 반드시 먼저 panel data 준비
    d = prepare_panel(df)

    if exclude_2020:
        d = d[d[YEAR] != 2020].copy()

    # lagged dependent variable
    d = d.sort_values([ID, YEAR])
    d["lag_RevPar_log"] = d.groupby(ID)[Y].shift(1)

    # lagged IT structure
    d, names = add_lagged_structure(d, it_col, risk_col, lag=1, standardize=True)

    needed = [
        Y, "lag_RevPar_log", ID, YEAR, risk_col,
        names["mean_col"], names["within_col"],
        names["risk_within_col"], names["w_int"], names["b_int"]
    ] + CONTROLS_term

    d = d.dropna(subset=needed).copy()

    # region / segment 더미화
    d = pd.get_dummies(d, columns=[REGION, SEG], drop_first=True)

    exog_cols = [
        "lag_RevPar_log",
        risk_col,
        names["within_col"],
        names["mean_col"],
        names["w_int"],
        names["b_int"]
    ] + CONTROLS_term

    exog_cols += [
        c for c in d.columns
        if c.startswith(REGION + "_") or c.startswith(SEG + "_")
    ]

    d = d.set_index([ID, YEAR])

    mod = PanelOLS(
        dependent=d[Y],
        exog=d[exog_cols],
        entity_effects=True,
        time_effects=True,
        drop_absorbed=True
    )

    res = mod.fit(cov_type="clustered", cluster_entity=True)

    return res, d, names

def extract_key_terms_mixed_ldv(res, names, risk_col, label):
    terms = [
        "lag_RevPar_log",
        risk_col,
        names["within_col"],
        names["mean_col"],
        names["w_int"],
        names["b_int"]
    ]
    out = []
    for t in terms:
        if t in res.params.index:
            out.append({
                "model": label,
                "term": t,
                "coef": res.params[t],
                "se": res.bse[t],
                "p": res.pvalues[t]
            })
    return pd.DataFrame(out)


def extract_key_terms_panel_ldv(res, names, risk_col, label):
    if res is None:
        return pd.DataFrame()

    terms = [
        "lag_RevPar_log",
        risk_col,
        names["within_col"],
        names["mean_col"],
        names["w_int"],
        names["b_int"]
    ]
    out = []
    for t in terms:
        if t in res.params.index:
            out.append({
                "model": label,
                "term": t,
                "coef": res.params[t],
                "se": res.std_errors[t],
                "p": res.pvalues[t]
            })
    return pd.DataFrame(out)

def add_lagged_structure_multi(df, it_cols, risk_col, lag=1, standardize=True):
    d = df.copy().sort_values([ID, YEAR])

    names = {
        "risk": risk_col,
        "risk_mean": f"{risk_col}_mean_hotel",
        "risk_within": f"{risk_col}_within_hotel",
        "it": {}
    }

    # risk decomposition
    d[names["risk_mean"]] = d.groupby(ID)[risk_col].transform("mean")
    d[names["risk_within"]] = d[risk_col] - d[names["risk_mean"]]

    if standardize:
        d[risk_col] = zscore_safe(d[risk_col])
        d[names["risk_mean"]] = zscore_safe(d[names["risk_mean"]])
        d[names["risk_within"]] = zscore_safe(d[names["risk_within"]])

    # each IT component
    for it_col in it_cols:
        lag_col = f"lag_{it_col}"
        mean_col = f"{lag_col}_mean_hotel"
        within_col = f"{lag_col}_within"
        w_int = f"{lag_col}_w_x_R"
        b_int = f"{lag_col}_b_x_Rw"

        d[lag_col] = d.groupby(ID)[it_col].shift(lag)
        d[mean_col] = d.groupby(ID)[lag_col].transform("mean")
        d[within_col] = d[lag_col] - d[mean_col]

        if standardize:
            d[lag_col] = zscore_safe(d[lag_col])
            d[mean_col] = zscore_safe(d[mean_col])
            d[within_col] = zscore_safe(d[within_col])

        d[w_int] = d[within_col] * d[risk_col]
        d[b_int] = d[mean_col] * d[names["risk_within"]]

        names["it"][it_col] = {
            "lag": lag_col,
            "mean": mean_col,
            "within": within_col,
            "w_int": w_int,
            "b_int": b_int
        }

    return d, names

def fit_mixedlm_joint(df, it_cols, risk_col, exclude_2020=False, random_slope_col=None, add_ldv=False):
    d = prepare_panel(df)

    if exclude_2020:
        d = d[d[YEAR] != 2020].copy()

    if add_ldv:
        d = d.sort_values([ID, YEAR])
        d["lag_RevPar_log"] = d.groupby(ID)[Y].shift(1)

    d, names = add_lagged_structure_multi(d, it_cols, risk_col, lag=1, standardize=True)

    rhs_terms = []
    needed = [Y, ID, REGION, SEG, YEAR, risk_col]

    if add_ldv:
        rhs_terms.append("lag_RevPar_log")
        needed.append("lag_RevPar_log")

    rhs_terms.append(risk_col)

    for it_col in it_cols:
        block = names["it"][it_col]
        rhs_terms += [block["within"], block["mean"], block["w_int"], block["b_int"]]
        needed += [block["within"], block["mean"], block["w_int"], block["b_int"]]

    needed += CONTROLS_term
    rhs_terms += CONTROLS_term
    rhs_terms += [f"C({SEG})", f"C({REGION})", f"C({YEAR})"]

    d = d.dropna(subset=needed).copy()

    d[SEG] = d[SEG].astype("category")
    d[REGION] = d[REGION].astype("category")
    d[YEAR] = d[YEAR].astype("category")

    formula = f"{Y} ~ " + " + ".join(rhs_terms)

    re_formula = "1"
    if random_slope_col is not None:
        re_formula = f"1 + {names['it'][random_slope_col]['within']}"

    model = smf.mixedlm(formula, data=d, groups=d[ID], re_formula=re_formula)

    try:
        res = model.fit(method="lbfgs", reml=True)
    except Exception:
        res = model.fit(method="powell", reml=True)

    return res, d, names


def fit_panel_fe_cluster_joint(df, it_cols, risk_col, exclude_2020=False, add_ldv=False):
    if not PANEL_OK:
        return None, None, None

    d = prepare_panel(df)

    if exclude_2020:
        d = d[d[YEAR] != 2020].copy()

    if add_ldv:
        d = d.sort_values([ID, YEAR])
        d["lag_RevPar_log"] = d.groupby(ID)[Y].shift(1)

    d, names = add_lagged_structure_multi(d, it_cols, risk_col, lag=1, standardize=True)

    needed = [Y, ID, YEAR, risk_col]
    exog_cols = []

    if add_ldv:
        needed.append("lag_RevPar_log")
        exog_cols.append("lag_RevPar_log")

    exog_cols.append(risk_col)

    for it_col in it_cols:
        block = names["it"][it_col]
        exog_cols += [block["within"], block["mean"], block["w_int"], block["b_int"]]
        needed += [block["within"], block["mean"], block["w_int"], block["b_int"]]

    needed += CONTROLS_term
    exog_cols += CONTROLS_term

    d = d.dropna(subset=needed).copy()

    d = pd.get_dummies(d, columns=[REGION, SEG], drop_first=True)
    exog_cols += [c for c in d.columns if c.startswith(REGION + "_") or c.startswith(SEG + "_")]

    d = d.set_index([ID, YEAR])

    mod = PanelOLS(
        dependent=d[Y],
        exog=d[exog_cols],
        entity_effects=True,
        time_effects=True,
        drop_absorbed=True
    )

    res = mod.fit(cov_type="clustered", cluster_entity=True)
    return res, d, names

def extract_key_terms_joint_mixed(res, names, risk_col, label, it_cols, add_ldv=False):
    terms = [risk_col]
    if add_ldv:
        terms.append("lag_RevPar_log")

    for it_col in it_cols:
        block = names["it"][it_col]
        terms += [block["within"], block["mean"], block["w_int"], block["b_int"]]

    out = []
    for t in terms:
        if t in res.params.index:
            out.append({
                "model": label,
                "term": t,
                "coef": res.params[t],
                "se": res.bse[t],
                "p": res.pvalues[t]
            })
    return pd.DataFrame(out)


def extract_key_terms_joint_panel(res, names, risk_col, label, it_cols, add_ldv=False):
    if res is None:
        return pd.DataFrame()

    terms = [risk_col]
    if add_ldv:
        terms.append("lag_RevPar_log")

    for it_col in it_cols:
        block = names["it"][it_col]
        terms += [block["within"], block["mean"], block["w_int"], block["b_int"]]

    out = []
    for t in terms:
        if t in res.params.index:
            out.append({
                "model": label,
                "term": t,
                "coef": res.params[t],
                "se": res.std_errors[t],
                "p": res.pvalues[t]
            })
    return pd.DataFrame(out)

In [44]:
#Tropical + IT total

risk_col = "Tropical_Risk_z"
it_col = "tr_ITTotalExpense"

# -----------------------------
# 기존 3개
# -----------------------------
res_full, d_full, names_full = fit_mixedlm_model(df, it_col, risk_col, exclude_2020=False, random_slope=True)
res_no20, d_no20, names_no20 = fit_mixedlm_model(df, it_col, risk_col, exclude_2020=True, random_slope=True)
res_fe, d_fe, names_fe = fit_panel_fe_cluster(df, it_col, risk_col, exclude_2020=False)

# -----------------------------
# LDV 포함 2개 추가
# -----------------------------
res_full_ldv, d_full_ldv, names_full_ldv = fit_mixedlm_model_ldv(df, it_col, risk_col, exclude_2020=False, random_slope=True)
res_fe_ldv, d_fe_ldv, names_fe_ldv = fit_panel_fe_cluster_ldv(df, it_col, risk_col, exclude_2020=False)

# -----------------------------
# 전체 계수표
# -----------------------------
full_mixed = extract_full_mixed_results(res_full)
full_mixed["model"] = "MixedLM_full"

full_mixed_no20 = extract_full_mixed_results(res_no20)
full_mixed_no20["model"] = "MixedLM_no2020"

full_panel = extract_full_panel_results(res_fe)
full_panel["model"] = "PanelFE_cluster"

full_mixed_ldv = extract_full_mixed_results(res_full_ldv)
full_mixed_ldv["model"] = "MixedLM_full_LDV"

full_panel_ldv = extract_full_panel_results(res_fe_ldv)
full_panel_ldv["model"] = "PanelFE_cluster_LDV"

full_long = pd.concat([
    full_mixed,
    full_mixed_no20,
    full_panel,
    full_mixed_ldv,
    full_panel_ldv
], ignore_index=True)

full_long["coef_se_p"] = full_long.apply(
    lambda x: f"{x['coef']:.3f} ({x['se']:.3f}), p={x['p']:.3f}",
    axis=1
)

full_wide = full_long.pivot_table(
    index="term",
    columns="model",
    values="coef_se_p",
    aggfunc="first"
)

# -----------------------------
# 핵심 계수표
# -----------------------------
key_coef = pd.concat([
    extract_key_terms_mixed(res_full, names_full, risk_col, "MixedLM_full"),
    extract_key_terms_mixed(res_no20, names_no20, risk_col, "MixedLM_no2020"),
    extract_key_terms_panel(res_fe, names_fe, risk_col, "PanelFE_cluster"),
    extract_key_terms_mixed_ldv(res_full_ldv, names_full_ldv, risk_col, "MixedLM_full_LDV"),
    extract_key_terms_panel_ldv(res_fe_ldv, names_fe_ldv, risk_col, "PanelFE_cluster_LDV")
], ignore_index=True)

pretty_key = pretty_compare(key_coef)

# -----------------------------
# 진단
# -----------------------------
vif = compute_vif_from_mixedlm_result(res_full)
vif_ldv = compute_vif_from_mixedlm_result(res_full_ldv)

r2_mixed = mixedlm_r2_pred(res_full)
r2_mixed_no20 = mixedlm_r2_pred(res_no20)
r2_mixed_ldv = mixedlm_r2_pred(res_full_ldv)

r2_panel = panel_r2_table(res_fe)
r2_panel_ldv = panel_r2_table(res_fe_ldv)

ar1_desc, lb_table = residual_autocorr_diagnostics(res_full, d_full)
ar1_desc_ldv, lb_table_ldv = residual_autocorr_diagnostics(res_full_ldv, d_full_ldv)

fit_stats = pd.DataFrame([
    {"model": "MixedLM_full", "nobs": res_full.nobs, "llf": res_full.llf, "aic": res_full.aic, "bic": res_full.bic},
    {"model": "MixedLM_no2020", "nobs": res_no20.nobs, "llf": res_no20.llf, "aic": res_no20.aic, "bic": res_no20.bic},
    {"model": "MixedLM_full_LDV", "nobs": res_full_ldv.nobs, "llf": res_full_ldv.llf, "aic": res_full_ldv.aic, "bic": res_full_ldv.bic},
    {"model": "PanelFE_cluster", "nobs": getattr(res_fe, "nobs", np.nan), "llf": np.nan, "aic": np.nan, "bic": np.nan},
    {"model": "PanelFE_cluster_LDV", "nobs": getattr(res_fe_ldv, "nobs", np.nan), "llf": np.nan, "aic": np.nan, "bic": np.nan},
])

print("\n================ KEY COEFFICIENTS ================")
print(pretty_key)

print("\n================ FIT STATS ================")
print(fit_stats)

print("\n================ VIF (BASE) ================")
print(vif.head(30))

print("\n================ VIF (LDV) ================")
print(vif_ldv.head(30))

print("\n================ R2 BASE ================")
print(r2_mixed)

print("\n================ R2 LDV ================")
print(r2_mixed_ldv)

print("\n================ AR1 BASE ================")
print(ar1_desc)
print(lb_table)

print("\n================ AR1 LDV ================")
print(ar1_desc_ldv)
print(lb_table_ldv)

with pd.ExcelWriter("scenario1_tropical_totalIT_with_LDV.xlsx") as writer:
    full_long.to_excel(writer, sheet_name="full_coef_long", index=False)
    full_wide.to_excel(writer, sheet_name="full_coef_wide")
    key_coef.to_excel(writer, sheet_name="key_coef", index=False)
    pretty_key.to_excel(writer, sheet_name="pretty_key")
    fit_stats.to_excel(writer, sheet_name="fit_stats", index=False)
    vif.to_excel(writer, sheet_name="vif_base", index=False)
    vif_ldv.to_excel(writer, sheet_name="vif_ldv", index=False)
    r2_mixed.to_excel(writer, sheet_name="r2_mixed_base", index=False)
    r2_mixed_no20.to_excel(writer, sheet_name="r2_mixed_no2020", index=False)
    r2_mixed_ldv.to_excel(writer, sheet_name="r2_mixed_ldv", index=False)
    r2_panel.to_excel(writer, sheet_name="r2_panel_base", index=False)
    r2_panel_ldv.to_excel(writer, sheet_name="r2_panel_ldv", index=False)
    ar1_desc.to_excel(writer, sheet_name="ar1_base")
    lb_table.to_excel(writer, sheet_name="ljung_box_base")
    ar1_desc_ldv.to_excel(writer, sheet_name="ar1_ldv")
    lb_table_ldv.to_excel(writer, sheet_name="ljung_box_ldv")

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1919/2419301966.py:221: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

lag_tr_ITTotalExpense_mean_hotel, noaa_region9_Midwest, noaa_region9_Northeast, noaa_region9_Northern Plains, noaa_region9_Northwest, noaa_region9_Pacific, noaa_region9_South, noaa_region9_Southwest, noaa_region9_West

  res = mod.fit(cov_type="clustered", cluster_entity=True)
/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model


================ KEY COEFFICIENTS ================
model                                        MixedLM_full  \
term                                                        
Tropical_Risk_z                    0.004 (0.001), p=0.000   
lag_RevPar_log                                        NaN   
lag_tr_ITTotalExpense_b_x_Rw      -0.002 (0.001), p=0.000   
lag_tr_ITTotalExpense_mean_hotel  -0.004 (0.004), p=0.237   
lag_tr_ITTotalExpense_w_x_R        0.002 (0.001), p=0.003   
lag_tr_ITTotalExpense_within       0.006 (0.001), p=0.000   

model                                    MixedLM_full_LDV  \
term                                                        
Tropical_Risk_z                    0.004 (0.001), p=0.000   
lag_RevPar_log                     0.205 (0.003), p=0.000   
lag_tr_ITTotalExpense_b_x_Rw      -0.003 (0.001), p=0.000   
lag_tr_ITTotalExpense_mean_hotel  -0.018 (0.003), p=0.000   
lag_tr_ITTotalExpense_w_x_R        0.001 (0.001), p=0.065   
lag_tr_ITTotalExpense_within    

In [45]:
#Fire + IT total

risk_col = "Fire_Risk_z"
it_col   = "tr_ITTotalExpense"

# -----------------------------
# 기존 3개
# -----------------------------
res_full, d_full, names_full = fit_mixedlm_model(df, it_col, risk_col, exclude_2020=False, random_slope=True)
res_no20, d_no20, names_no20 = fit_mixedlm_model(df, it_col, risk_col, exclude_2020=True, random_slope=True)
res_fe, d_fe, names_fe = fit_panel_fe_cluster(df, it_col, risk_col, exclude_2020=False)

# -----------------------------
# LDV 포함 2개 추가
# -----------------------------
res_full_ldv, d_full_ldv, names_full_ldv = fit_mixedlm_model_ldv(df, it_col, risk_col, exclude_2020=False, random_slope=True)
res_fe_ldv, d_fe_ldv, names_fe_ldv = fit_panel_fe_cluster_ldv(df, it_col, risk_col, exclude_2020=False)

# -----------------------------
# 전체 계수표
# -----------------------------
full_mixed = extract_full_mixed_results(res_full)
full_mixed["model"] = "MixedLM_full"

full_mixed_no20 = extract_full_mixed_results(res_no20)
full_mixed_no20["model"] = "MixedLM_no2020"

full_panel = extract_full_panel_results(res_fe)
full_panel["model"] = "PanelFE_cluster"

full_mixed_ldv = extract_full_mixed_results(res_full_ldv)
full_mixed_ldv["model"] = "MixedLM_full_LDV"

full_panel_ldv = extract_full_panel_results(res_fe_ldv)
full_panel_ldv["model"] = "PanelFE_cluster_LDV"

full_long = pd.concat([
    full_mixed,
    full_mixed_no20,
    full_panel,
    full_mixed_ldv,
    full_panel_ldv
], ignore_index=True)

full_long["coef_se_p"] = full_long.apply(
    lambda x: f"{x['coef']:.3f} ({x['se']:.3f}), p={x['p']:.3f}",
    axis=1
)

full_wide = full_long.pivot_table(
    index="term",
    columns="model",
    values="coef_se_p",
    aggfunc="first"
)

# -----------------------------
# 핵심 계수표
# -----------------------------
key_coef = pd.concat([
    extract_key_terms_mixed(res_full, names_full, risk_col, "MixedLM_full"),
    extract_key_terms_mixed(res_no20, names_no20, risk_col, "MixedLM_no2020"),
    extract_key_terms_panel(res_fe, names_fe, risk_col, "PanelFE_cluster"),
    extract_key_terms_mixed_ldv(res_full_ldv, names_full_ldv, risk_col, "MixedLM_full_LDV"),
    extract_key_terms_panel_ldv(res_fe_ldv, names_fe_ldv, risk_col, "PanelFE_cluster_LDV")
], ignore_index=True)

pretty_key = pretty_compare(key_coef)

# -----------------------------
# 진단
# -----------------------------
vif = compute_vif_from_mixedlm_result(res_full)
vif_ldv = compute_vif_from_mixedlm_result(res_full_ldv)

r2_mixed = mixedlm_r2_pred(res_full)
r2_mixed_no20 = mixedlm_r2_pred(res_no20)
r2_mixed_ldv = mixedlm_r2_pred(res_full_ldv)

r2_panel = panel_r2_table(res_fe)
r2_panel_ldv = panel_r2_table(res_fe_ldv)

ar1_desc, lb_table = residual_autocorr_diagnostics(res_full, d_full)
ar1_desc_ldv, lb_table_ldv = residual_autocorr_diagnostics(res_full_ldv, d_full_ldv)

fit_stats = pd.DataFrame([
    {"model": "MixedLM_full", "nobs": res_full.nobs, "llf": res_full.llf, "aic": res_full.aic, "bic": res_full.bic},
    {"model": "MixedLM_no2020", "nobs": res_no20.nobs, "llf": res_no20.llf, "aic": res_no20.aic, "bic": res_no20.bic},
    {"model": "MixedLM_full_LDV", "nobs": res_full_ldv.nobs, "llf": res_full_ldv.llf, "aic": res_full_ldv.aic, "bic": res_full_ldv.bic},
    {"model": "PanelFE_cluster", "nobs": getattr(res_fe, "nobs", np.nan), "llf": np.nan, "aic": np.nan, "bic": np.nan},
    {"model": "PanelFE_cluster_LDV", "nobs": getattr(res_fe_ldv, "nobs", np.nan), "llf": np.nan, "aic": np.nan, "bic": np.nan},
])

print("\n================ KEY COEFFICIENTS ================")
print(pretty_key)

print("\n================ FIT STATS ================")
print(fit_stats)

print("\n================ VIF (BASE) ================")
print(vif.head(30))

print("\n================ VIF (LDV) ================")
print(vif_ldv.head(30))

print("\n================ R2 BASE ================")
print(r2_mixed)

print("\n================ R2 LDV ================")
print(r2_mixed_ldv)

print("\n================ AR1 BASE ================")
print(ar1_desc)
print(lb_table)

print("\n================ AR1 LDV ================")
print(ar1_desc_ldv)
print(lb_table_ldv)

with pd.ExcelWriter("scenario1_tropical_totalIT_with_LDV.xlsx") as writer:
    full_long.to_excel(writer, sheet_name="full_coef_long", index=False)
    full_wide.to_excel(writer, sheet_name="full_coef_wide")
    key_coef.to_excel(writer, sheet_name="key_coef", index=False)
    pretty_key.to_excel(writer, sheet_name="pretty_key")
    fit_stats.to_excel(writer, sheet_name="fit_stats", index=False)
    vif.to_excel(writer, sheet_name="vif_base", index=False)
    vif_ldv.to_excel(writer, sheet_name="vif_ldv", index=False)
    r2_mixed.to_excel(writer, sheet_name="r2_mixed_base", index=False)
    r2_mixed_no20.to_excel(writer, sheet_name="r2_mixed_no2020", index=False)
    r2_mixed_ldv.to_excel(writer, sheet_name="r2_mixed_ldv", index=False)
    r2_panel.to_excel(writer, sheet_name="r2_panel_base", index=False)
    r2_panel_ldv.to_excel(writer, sheet_name="r2_panel_ldv", index=False)
    ar1_desc.to_excel(writer, sheet_name="ar1_base")
    lb_table.to_excel(writer, sheet_name="ljung_box_base")
    ar1_desc_ldv.to_excel(writer, sheet_name="ar1_ldv")
    lb_table_ldv.to_excel(writer, sheet_name="ljung_box_ldv")

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1919/2419301966.py:221: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

lag_tr_ITTotalExpense_mean_hotel, noaa_region9_Midwest, noaa_region9_Northeast, noaa_region9_Northern Plains, noaa_region9_Northwest, noaa_region9_Pacific, noaa_region9_South, noaa_region9_Southwest, noaa_region9_West

  res = mod.fit(cov_type="clustered", cluster_entity=True)
/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model


================ KEY COEFFICIENTS ================
model                                        MixedLM_full  \
term                                                        
Fire_Risk_z                        0.000 (0.001), p=0.993   
lag_RevPar_log                                        NaN   
lag_tr_ITTotalExpense_b_x_Rw      -0.003 (0.001), p=0.000   
lag_tr_ITTotalExpense_mean_hotel  -0.005 (0.004), p=0.215   
lag_tr_ITTotalExpense_w_x_R        0.002 (0.001), p=0.001   
lag_tr_ITTotalExpense_within       0.007 (0.001), p=0.000   

model                                    MixedLM_full_LDV  \
term                                                        
Fire_Risk_z                        0.001 (0.001), p=0.478   
lag_RevPar_log                     0.204 (0.003), p=0.000   
lag_tr_ITTotalExpense_b_x_Rw      -0.003 (0.001), p=0.000   
lag_tr_ITTotalExpense_mean_hotel  -0.019 (0.003), p=0.000   
lag_tr_ITTotalExpense_w_x_R        0.001 (0.001), p=0.306   
lag_tr_ITTotalExpense_within    

In [48]:
risk_col = "Tropical_Risk_z"
it_cols = ["tr_ITTotalLabor", "tr_ITService", "tr_ITSystem"]

# random slope는 labor within에만
random_slope_col = "tr_ITTotalLabor"

# -------------------------------------------------
# 1. Base models
# -------------------------------------------------
res_full, d_full, names_full = fit_mixedlm_joint(
    df, it_cols, risk_col,
    exclude_2020=False,
    random_slope_col=random_slope_col,
    add_ldv=False
)

res_no20, d_no20, names_no20 = fit_mixedlm_joint(
    df, it_cols, risk_col,
    exclude_2020=True,
    random_slope_col=random_slope_col,
    add_ldv=False
)

res_fe, d_fe, names_fe = fit_panel_fe_cluster_joint(
    df, it_cols, risk_col,
    exclude_2020=False,
    add_ldv=False
)

# -------------------------------------------------
# 2. LDV models
# -------------------------------------------------
res_full_ldv, d_full_ldv, names_full_ldv = fit_mixedlm_joint(
    df, it_cols, risk_col,
    exclude_2020=False,
    random_slope_col=random_slope_col,
    add_ldv=True
)

res_fe_ldv, d_fe_ldv, names_fe_ldv = fit_panel_fe_cluster_joint(
    df, it_cols, risk_col,
    exclude_2020=False,
    add_ldv=True
)

# -------------------------------------------------
# 3. Full coefficient tables
# -------------------------------------------------
full_mixed = extract_full_mixed_results(res_full)
full_mixed["model"] = "MixedLM_full"

full_mixed_no20 = extract_full_mixed_results(res_no20)
full_mixed_no20["model"] = "MixedLM_no2020"

full_panel = extract_full_panel_results(res_fe)
full_panel["model"] = "PanelFE_cluster"

full_mixed_ldv = extract_full_mixed_results(res_full_ldv)
full_mixed_ldv["model"] = "MixedLM_full_LDV"

full_panel_ldv = extract_full_panel_results(res_fe_ldv)
full_panel_ldv["model"] = "PanelFE_cluster_LDV"

full_long = pd.concat([
    full_mixed, full_mixed_no20, full_panel, full_mixed_ldv, full_panel_ldv
], ignore_index=True)

full_long["coef_se_p"] = full_long.apply(
    lambda x: f"{x['coef']:.3f} ({x['se']:.3f}), p={x['p']:.3f}",
    axis=1
)

full_wide = full_long.pivot_table(
    index="term",
    columns="model",
    values="coef_se_p",
    aggfunc="first"
)

# -------------------------------------------------
# 4. Key coefficient tables
# -------------------------------------------------
key_coef = pd.concat([
    extract_key_terms_joint_mixed(res_full, names_full, risk_col, "MixedLM_full", it_cols, add_ldv=False),
    extract_key_terms_joint_mixed(res_no20, names_no20, risk_col, "MixedLM_no2020", it_cols, add_ldv=False),
    extract_key_terms_joint_panel(res_fe, names_fe, risk_col, "PanelFE_cluster", it_cols, add_ldv=False),
    extract_key_terms_joint_mixed(res_full_ldv, names_full_ldv, risk_col, "MixedLM_full_LDV", it_cols, add_ldv=True),
    extract_key_terms_joint_panel(res_fe_ldv, names_fe_ldv, risk_col, "PanelFE_cluster_LDV", it_cols, add_ldv=True)
], ignore_index=True)

pretty_key = pretty_compare(key_coef)

# -------------------------------------------------
# 5. Diagnostics
# -------------------------------------------------
vif_base = compute_vif_from_mixedlm_result(res_full)
vif_ldv = compute_vif_from_mixedlm_result(res_full_ldv)

r2_base = mixedlm_r2_pred(res_full)
r2_no20 = mixedlm_r2_pred(res_no20)
r2_ldv = mixedlm_r2_pred(res_full_ldv)

r2_panel = panel_r2_table(res_fe)
r2_panel_ldv = panel_r2_table(res_fe_ldv)

ar1_desc, lb_table = residual_autocorr_diagnostics(res_full, d_full)
ar1_desc_ldv, lb_table_ldv = residual_autocorr_diagnostics(res_full_ldv, d_full_ldv)

fit_stats = pd.DataFrame([
    {"model": "MixedLM_full", "nobs": res_full.nobs, "llf": res_full.llf, "aic": res_full.aic, "bic": res_full.bic},
    {"model": "MixedLM_no2020", "nobs": res_no20.nobs, "llf": res_no20.llf, "aic": res_no20.aic, "bic": res_no20.bic},
    {"model": "MixedLM_full_LDV", "nobs": res_full_ldv.nobs, "llf": res_full_ldv.llf, "aic": res_full_ldv.aic, "bic": res_full_ldv.bic},
    {"model": "PanelFE_cluster", "nobs": getattr(res_fe, "nobs", np.nan), "llf": np.nan, "aic": np.nan, "bic": np.nan},
    {"model": "PanelFE_cluster_LDV", "nobs": getattr(res_fe_ldv, "nobs", np.nan), "llf": np.nan, "aic": np.nan, "bic": np.nan},
])

print("\n================ KEY COEFFICIENTS ================")
print(pretty_key)

print("\n================ FIT STATS ================")
print(fit_stats)

print("\n================ VIF BASE ================")
print(vif_base.head(40))

print("\n================ VIF LDV ================")
print(vif_ldv.head(40))

print("\n================ R2 BASE ================")
print(r2_base)

print("\n================ R2 LDV ================")
print(r2_ldv)

print("\n================ AR1 BASE ================")
print(ar1_desc)
print(lb_table)

print("\n================ AR1 LDV ================")
print(ar1_desc_ldv)
print(lb_table_ldv)

with pd.ExcelWriter("scenario3_tropical_joint_decomp_with_LDV.xlsx") as writer:
    full_long.to_excel(writer, sheet_name="full_coef_long", index=False)
    full_wide.to_excel(writer, sheet_name="full_coef_wide")
    key_coef.to_excel(writer, sheet_name="key_coef", index=False)
    pretty_key.to_excel(writer, sheet_name="pretty_key")
    fit_stats.to_excel(writer, sheet_name="fit_stats", index=False)
    vif_base.to_excel(writer, sheet_name="vif_base", index=False)
    vif_ldv.to_excel(writer, sheet_name="vif_ldv", index=False)
    r2_base.to_excel(writer, sheet_name="r2_base", index=False)
    r2_no20.to_excel(writer, sheet_name="r2_no2020", index=False)
    r2_ldv.to_excel(writer, sheet_name="r2_ldv", index=False)
    r2_panel.to_excel(writer, sheet_name="r2_panel", index=False)
    r2_panel_ldv.to_excel(writer, sheet_name="r2_panel_ldv", index=False)
    ar1_desc.to_excel(writer, sheet_name="ar1_base")
    lb_table.to_excel(writer, sheet_name="ljung_box_base")
    ar1_desc_ldv.to_excel(writer, sheet_name="ar1_ldv")
    lb_table_ldv.to_excel(writer, sheet_name="ljung_box_ldv")

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1919/4245025718.py:722: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

lag_tr_ITTotalLabor_mean_hotel, lag_tr_ITService_mean_hotel, lag_tr_ITSystem_mean_hotel, noaa_region9_Midwest, noaa_region9_Northeast, noaa_region9_Northern Plains, noaa_region9_Northwest, noaa_region9_Pacific, noaa_region9_South, noaa_region9_Southwest, noaa_region9_West

  res = mod.fit(cov_type="clustered", cluster_entity=True)
/Users/jheo/opt/anaconda3/lib/python3.13/


================ KEY COEFFICIENTS ================
model                                      MixedLM_full  \
term                                                      
Tropical_Risk_z                  0.006 (0.005), p=0.228   
lag_RevPar_log                                      NaN   
lag_tr_ITService_b_x_Rw          0.002 (0.004), p=0.615   
lag_tr_ITService_mean_hotel     -0.029 (0.006), p=0.000   
lag_tr_ITService_w_x_R          -0.000 (0.002), p=0.968   
lag_tr_ITService_within         -0.003 (0.002), p=0.173   
lag_tr_ITSystem_b_x_Rw          -0.006 (0.005), p=0.165   
lag_tr_ITSystem_mean_hotel      -0.062 (0.011), p=0.000   
lag_tr_ITSystem_w_x_R           -0.003 (0.002), p=0.231   
lag_tr_ITSystem_within           0.000 (0.003), p=0.955   
lag_tr_ITTotalLabor_b_x_Rw      -0.004 (0.006), p=0.482   
lag_tr_ITTotalLabor_mean_hotel  -0.012 (0.009), p=0.165   
lag_tr_ITTotalLabor_w_x_R       -0.001 (0.002), p=0.702   
lag_tr_ITTotalLabor_within       0.002 (0.002), p=0.439   

mod

In [49]:
#Fire + IT 3 decompisition

risk_col = "Fire_Risk_z"
it_cols = ["tr_ITTotalLabor", "tr_ITService", "tr_ITSystem"]

# random slope는 labor within에만
random_slope_col = "tr_ITTotalLabor"

# -------------------------------------------------
# 1. Base models
# -------------------------------------------------
res_full, d_full, names_full = fit_mixedlm_joint(
    df, it_cols, risk_col,
    exclude_2020=False,
    random_slope_col=random_slope_col,
    add_ldv=False
)

res_no20, d_no20, names_no20 = fit_mixedlm_joint(
    df, it_cols, risk_col,
    exclude_2020=True,
    random_slope_col=random_slope_col,
    add_ldv=False
)

res_fe, d_fe, names_fe = fit_panel_fe_cluster_joint(
    df, it_cols, risk_col,
    exclude_2020=False,
    add_ldv=False
)

# -------------------------------------------------
# 2. LDV models
# -------------------------------------------------
res_full_ldv, d_full_ldv, names_full_ldv = fit_mixedlm_joint(
    df, it_cols, risk_col,
    exclude_2020=False,
    random_slope_col=random_slope_col,
    add_ldv=True
)

res_fe_ldv, d_fe_ldv, names_fe_ldv = fit_panel_fe_cluster_joint(
    df, it_cols, risk_col,
    exclude_2020=False,
    add_ldv=True
)

# -------------------------------------------------
# 3. Full coefficient tables
# -------------------------------------------------
full_mixed = extract_full_mixed_results(res_full)
full_mixed["model"] = "MixedLM_full"

full_mixed_no20 = extract_full_mixed_results(res_no20)
full_mixed_no20["model"] = "MixedLM_no2020"

full_panel = extract_full_panel_results(res_fe)
full_panel["model"] = "PanelFE_cluster"

full_mixed_ldv = extract_full_mixed_results(res_full_ldv)
full_mixed_ldv["model"] = "MixedLM_full_LDV"

full_panel_ldv = extract_full_panel_results(res_fe_ldv)
full_panel_ldv["model"] = "PanelFE_cluster_LDV"

full_long = pd.concat([
    full_mixed, full_mixed_no20, full_panel, full_mixed_ldv, full_panel_ldv
], ignore_index=True)

full_long["coef_se_p"] = full_long.apply(
    lambda x: f"{x['coef']:.3f} ({x['se']:.3f}), p={x['p']:.3f}",
    axis=1
)

full_wide = full_long.pivot_table(
    index="term",
    columns="model",
    values="coef_se_p",
    aggfunc="first"
)

# -------------------------------------------------
# 4. Key coefficient tables
# -------------------------------------------------
key_coef = pd.concat([
    extract_key_terms_joint_mixed(res_full, names_full, risk_col, "MixedLM_full", it_cols, add_ldv=False),
    extract_key_terms_joint_mixed(res_no20, names_no20, risk_col, "MixedLM_no2020", it_cols, add_ldv=False),
    extract_key_terms_joint_panel(res_fe, names_fe, risk_col, "PanelFE_cluster", it_cols, add_ldv=False),
    extract_key_terms_joint_mixed(res_full_ldv, names_full_ldv, risk_col, "MixedLM_full_LDV", it_cols, add_ldv=True),
    extract_key_terms_joint_panel(res_fe_ldv, names_fe_ldv, risk_col, "PanelFE_cluster_LDV", it_cols, add_ldv=True)
], ignore_index=True)

pretty_key = pretty_compare(key_coef)

# -------------------------------------------------
# 5. Diagnostics
# -------------------------------------------------
vif_base = compute_vif_from_mixedlm_result(res_full)
vif_ldv = compute_vif_from_mixedlm_result(res_full_ldv)

r2_base = mixedlm_r2_pred(res_full)
r2_no20 = mixedlm_r2_pred(res_no20)
r2_ldv = mixedlm_r2_pred(res_full_ldv)

r2_panel = panel_r2_table(res_fe)
r2_panel_ldv = panel_r2_table(res_fe_ldv)

ar1_desc, lb_table = residual_autocorr_diagnostics(res_full, d_full)
ar1_desc_ldv, lb_table_ldv = residual_autocorr_diagnostics(res_full_ldv, d_full_ldv)

fit_stats = pd.DataFrame([
    {"model": "MixedLM_full", "nobs": res_full.nobs, "llf": res_full.llf, "aic": res_full.aic, "bic": res_full.bic},
    {"model": "MixedLM_no2020", "nobs": res_no20.nobs, "llf": res_no20.llf, "aic": res_no20.aic, "bic": res_no20.bic},
    {"model": "MixedLM_full_LDV", "nobs": res_full_ldv.nobs, "llf": res_full_ldv.llf, "aic": res_full_ldv.aic, "bic": res_full_ldv.bic},
    {"model": "PanelFE_cluster", "nobs": getattr(res_fe, "nobs", np.nan), "llf": np.nan, "aic": np.nan, "bic": np.nan},
    {"model": "PanelFE_cluster_LDV", "nobs": getattr(res_fe_ldv, "nobs", np.nan), "llf": np.nan, "aic": np.nan, "bic": np.nan},
])

print("\n================ KEY COEFFICIENTS ================")
print(pretty_key)

print("\n================ FIT STATS ================")
print(fit_stats)

print("\n================ VIF BASE ================")
print(vif_base.head(40))

print("\n================ VIF LDV ================")
print(vif_ldv.head(40))

print("\n================ R2 BASE ================")
print(r2_base)

print("\n================ R2 LDV ================")
print(r2_ldv)

print("\n================ AR1 BASE ================")
print(ar1_desc)
print(lb_table)

print("\n================ AR1 LDV ================")
print(ar1_desc_ldv)
print(lb_table_ldv)

with pd.ExcelWriter("scenario3_tropical_joint_decomp_with_LDV.xlsx") as writer:
    full_long.to_excel(writer, sheet_name="full_coef_long", index=False)
    full_wide.to_excel(writer, sheet_name="full_coef_wide")
    key_coef.to_excel(writer, sheet_name="key_coef", index=False)
    pretty_key.to_excel(writer, sheet_name="pretty_key")
    fit_stats.to_excel(writer, sheet_name="fit_stats", index=False)
    vif_base.to_excel(writer, sheet_name="vif_base", index=False)
    vif_ldv.to_excel(writer, sheet_name="vif_ldv", index=False)
    r2_base.to_excel(writer, sheet_name="r2_base", index=False)
    r2_no20.to_excel(writer, sheet_name="r2_no2020", index=False)
    r2_ldv.to_excel(writer, sheet_name="r2_ldv", index=False)
    r2_panel.to_excel(writer, sheet_name="r2_panel", index=False)
    r2_panel_ldv.to_excel(writer, sheet_name="r2_panel_ldv", index=False)
    ar1_desc.to_excel(writer, sheet_name="ar1_base")
    lb_table.to_excel(writer, sheet_name="ljung_box_base")
    ar1_desc_ldv.to_excel(writer, sheet_name="ar1_ldv")
    lb_table_ldv.to_excel(writer, sheet_name="ljung_box_ldv")

/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2237: ConvergenceWarning: The MLE may be on the boundary of the parameter space.
  warnings.warn(msg, ConvergenceWarning)
/Users/jheo/opt/anaconda3/lib/python3.13/site-packages/statsmodels/regression/mixed_linear_model.py:2261: ConvergenceWarning: The Hessian matrix at the estimated parameter values is not positive definite.
  warnings.warn(msg, ConvergenceWarning)
/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1919/4245025718.py:722: AbsorbingEffectWarning: 
Variables have been fully absorbed and have removed from the regression:

lag_tr_ITTotalLabor_mean_hotel, lag_tr_ITService_mean_hotel, lag_tr_ITSystem_mean_hotel, noaa_region9_Midwest, noaa_region9_Nort


================ KEY COEFFICIENTS ================
model                                      MixedLM_full  \
term                                                      
Fire_Risk_z                      0.005 (0.006), p=0.392   
lag_RevPar_log                                      NaN   
lag_tr_ITService_b_x_Rw          0.004 (0.003), p=0.289   
lag_tr_ITService_mean_hotel     -0.027 (0.006), p=0.000   
lag_tr_ITService_w_x_R          -0.004 (0.001), p=0.003   
lag_tr_ITService_within         -0.003 (0.002), p=0.187   
lag_tr_ITSystem_b_x_Rw          -0.008 (0.005), p=0.072   
lag_tr_ITSystem_mean_hotel      -0.064 (0.011), p=0.000   
lag_tr_ITSystem_w_x_R            0.008 (0.002), p=0.000   
lag_tr_ITSystem_within          -0.001 (0.003), p=0.817   
lag_tr_ITTotalLabor_b_x_Rw       0.007 (0.006), p=0.231   
lag_tr_ITTotalLabor_mean_hotel  -0.010 (0.009), p=0.226   
lag_tr_ITTotalLabor_w_x_R       -0.001 (0.002), p=0.502   
lag_tr_ITTotalLabor_within       0.002 (0.002), p=0.418   

mod